# MultiMend — Kaggle A→Z baseline reproduction (first benchmark: QuixBugs-Python)

**Scope:** official MultiMend repository + official fine-tuned checkpoints + official QuixBugs benchmark.

This notebook deliberately does **not** run all six benchmarks. It first validates the complete pipeline on QuixBugs-Python:

`buggy/correct programs → official bug-line/context extraction → local RAG → CodeT5 checkpoint ensemble → 100 candidates/checkpoint → combine/deduplicate → patch validation → CSV/JSON summary`

No OpenAI/Claude/Gemini API is used. The model and the RAG embedding model are downloaded locally from Hugging Face.


Giai đoạn 0: 

In [ ]:
# Cell 1 — Kaggle environment
import os, sys, subprocess, textwrap, json, pathlib, shutil, time
from pathlib import Path

print("Python:", sys.version)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch import error:", repr(e))


In [ ]:
# Cell 2 — Clone the MultiMend28 fork.
# We intentionally do not recurse through all benchmark submodules yet.

REPO_URL = "https://github.com/ThaoTo2806/MultiMend28.git"
REPO = Path("/kaggle/working/MultiMend28")

if REPO.exists():
    print("Repo already exists:", REPO)
else:
    subprocess.run([
        "git", "clone", "--depth", "1",
        REPO_URL,
        str(REPO)
    ], check=True)

os.chdir(REPO)
print("HEAD:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Repo:", REPO)
print("Remote:", subprocess.check_output(["git", "remote", "get-url", "origin"], text=True).strip())
print("Top-level:", [p.name for p in REPO.iterdir()])


In [ ]:
# Cell 3 — Inspect the official repo BEFORE running anything
from pathlib import Path

for p in sorted((REPO / "src").rglob("*.py")):
    print(p.relative_to(REPO))

print("\nrequirements.txt:")
print((REPO / "requirements.txt").read_text())


In [ ]:
# Cell 4 — Install official MultiMend dependencies.
# Do NOT upgrade unrelated Kaggle packages.

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "-r",
        "requirements.txt",
        "more-itertools",
        "pygments",
    ],
    text=True,
)

if result.returncode != 0:
    raise RuntimeError("MultiMend dependency installation failed.")

print("MultiMend dependency installation complete.")

In [ ]:
# Cell 4b — Force the versions required by MultiMend
import subprocess
import sys

packages = [
    "numpy==1.26.4",
    "pandas==2.2.3",
    "torch==2.4.0",
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        *packages,
    ],
    check=True,
)

print("Pinned NumPy/Pandas/Torch versions installed.")

In [ ]:
# Cell 4c — Match torchvision with MultiMend's torch 2.4.0

import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        "torchvision==0.19.0",
    ],
    check=True,
)

print("torchvision 0.19.0 installed.")

In [16]:
# Fix NumPy for ChromaDB 0.5.5 / MultiMend
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m", "pip", "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        "numpy==1.26.4",
    ],
    check=True,
)

print("NumPy 1.26.4 installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 332.8 MB/s eta 0:00:00
NumPy 1.26.4 installed.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
datasets 2.21.0 requires fsspec[http]<=2024.6.1,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.2.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 wh

In [17]:
import numpy as np

print("NumPy:", np.__version__)


NumPy: 1.26.4


In [18]:
import torch
import transformers
import datasets
import chromadb
import sentence_transformers
import tree_sitter
import unidiff
import more_itertools
import pygments

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("chromadb:", chromadb.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("tree-sitter:", tree_sitter.__version__ if hasattr(tree_sitter, "__version__") else "unknown")
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


torch: 2.4.0+cu121
transformers: 4.44.2
datasets: 2.21.0
chromadb: 0.5.5
sentence-transformers: 3.0.1
tree-sitter: unknown
CUDA: True
GPU: Tesla T4


In [19]:
# Cell 5 — Verify versions and direct imports
import torch, transformers, datasets, chromadb, sentence_transformers
import tree_sitter, unidiff, more_itertools, pygments

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("chromadb:", chromadb.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("tree-sitter:", tree_sitter.__version__ if hasattr(tree_sitter, "__version__") else "unknown")
print("CUDA:", torch.cuda.is_available())


torch: 2.4.0+cu121
transformers: 4.44.2
datasets: 2.21.0
chromadb: 0.5.5
sentence-transformers: 3.0.1
tree-sitter: unknown
CUDA: True


In [25]:
from pathlib import Path

REPO = Path("/kaggle/working/MultiMend28")

print("Repo exists:", REPO.exists())
print("Repo:", REPO)

if REPO.exists():
    print("Contents:")
    for p in list(REPO.iterdir())[:20]:
        print(" -", p.name)


Repo exists: True
Repo: /kaggle/working/MultiMend28
Contents:
 - MultiMend.png
 - .gitattributes
 - src
 - .git
 - .gitmodules
 - README.md
 - .python-version
 - .gitignore
 - results
 - benchmarks
 - LICENSE
 - requirements.txt


In [27]:
# Cell 6 — Download QuixBugs

from pathlib import Path
import subprocess
import shutil

REPO = Path("/kaggle/working/MultiMend28")
QB = REPO / "benchmarks" / "QuixBugs"

if not REPO.exists():
    raise FileNotFoundError(
        f"MultiMend repo not found at {REPO}. "
        "Please run the clone cell first."
    )

if not (QB / "python_programs").exists():
    if QB.exists():
        shutil.rmtree(QB)

    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/jkoppel/QuixBugs.git",
        str(QB)
    ], check=True)

print("QuixBugs files:")
for d in ["python_programs", "correct_python_programs", "python_testcases"]:
    p = QB / d
    print(
        d,
        "=>", p.exists(),
        "count=", len(list(p.glob("*.py"))) if p.exists() else 0
    )


Cloning into '/kaggle/working/MultiMend28/benchmarks/QuixBugs'...


QuixBugs files:
python_programs => True count= 50
correct_python_programs => True count= 50
python_testcases => True count= 42


In [29]:
# ============================================================
# CELL 38 — PREPARE CHECKPOINTS FOR INFERENCE
# ============================================================
# Mục đích:
# - Chuẩn bị 5 checkpoint cuối để inference.
# - Copy checkpoint từ Kaggle Dataset vào thư mục project.
# - Đọc trainer_state.json để xác nhận global_step THỰC TẾ.
# - Không suy luận checkpoint ensemble chỉ từ tên thư mục.
#
# Các checkpoint được sử dụng:
#   348600
#   406700
#   464800
#   522900
#   581008
# ============================================================


import json
import shutil
from pathlib import Path


# ------------------------------------------------------------
# 1. Xác định thư mục project
# ------------------------------------------------------------

REPO = Path(
    "/kaggle/working/MultiMend28"
)


# ------------------------------------------------------------
# 2. Thư mục chứa các checkpoint dùng cho inference
# ------------------------------------------------------------

MODEL_DIR = (
    REPO
    / "models"
    / "multimend-codet5-small"
)

# Tạo thư mục nếu chưa tồn tại
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 3. Khai báo đường dẫn checkpoint nguồn
#
# Key:
#   global step dự kiến
#
# Value:
#   đường dẫn checkpoint trên Kaggle Dataset
# ------------------------------------------------------------

checkpoint_paths = {

    348600:
        "/kaggle/input/datasets/tramtvt/"
        "checkpoint348600/checkpoint-348600",

    406700:
        "/kaggle/input/datasets/tramtvt/"
        "checkpoint406700/checkpoint-406700",

    464800:
        "/kaggle/input/datasets/nguynhongbchtrm/"
        "checkpoint464800/checkpoint-464800",

    522900:
        "/kaggle/input/datasets/nguynhongbchtrm/"
        "checkpoint522900/checkpoint-522900",

    581008:
        "/kaggle/input/datasets/nguynhongbchtrm/"
        "checkpoint581008/checkpoint-581008",
}


# ------------------------------------------------------------
# 4. Copy từng checkpoint
# ------------------------------------------------------------

for step, source_text in checkpoint_paths.items():

    # Đường dẫn checkpoint nguồn
    source = Path(source_text)

    # Đường dẫn checkpoint đích
    target = (
        MODEL_DIR
        / f"checkpoint-{step}"
    )


    # --------------------------------------------------------
    # Kiểm tra checkpoint nguồn có tồn tại
    # --------------------------------------------------------

    if not source.is_dir():

        raise FileNotFoundError(
            f"Không tìm thấy checkpoint: {source}"
        )


    # --------------------------------------------------------
    # Nếu checkpoint đích đã tồn tại:
    # xóa để copy lại từ nguồn cho sạch.
    # --------------------------------------------------------

    if target.exists():

        shutil.rmtree(target)


    # --------------------------------------------------------
    # Copy toàn bộ checkpoint
    # --------------------------------------------------------

    shutil.copytree(
        source,
        target
    )


    # --------------------------------------------------------
    # 5. Đọc trainer_state.json
    #
    # Đây là bước quan trọng:
    # không chỉ tin vào tên checkpoint,
    # mà xác nhận global_step thực tế.
    # --------------------------------------------------------

    state_file = (
        target
        / "trainer_state.json"
    )


    with state_file.open(
        encoding="utf-8"
    ) as file:

        state = json.load(file)


    # --------------------------------------------------------
    # 6. In global_step thực tế
    # --------------------------------------------------------

    print(
        target.name,
        "global_step =",
        state.get("global_step")
    )


# ------------------------------------------------------------
# 7. Hoàn thành bước chuẩn bị checkpoint
# ------------------------------------------------------------

print(
    "Đã chuẩn bị checkpoint inference."
)


checkpoint-348600 global_step = 348600
checkpoint-406700 global_step = 406700
checkpoint-464800 global_step = 464800
checkpoint-522900 global_step = 522900
checkpoint-581008 global_step = 581008
Đã chuẩn bị checkpoint inference.


In [31]:
# ============================================================
# CELL 39 — CHECK 5 CHECKPOINTS + MODEL CONFIG
# ============================================================
# Mục đích:
# - Xác nhận 5 checkpoint đã được copy đúng.
# - Đọc config.json của từng checkpoint.
# - Kiểm tra các token ID cần thiết cho CodeT5.
# - Đặc biệt kiểm tra decoder_start_token_id vì lỗi
#   trước đó xảy ra tại model.generate().
# ============================================================

import json
from pathlib import Path

MODEL_DIR = Path(
    "/kaggle/working/MultiMend28/models/multimend-codet5-small"
)

for ckpt in sorted(MODEL_DIR.glob("checkpoint-*")):

    print(f"\n{'=' * 60}")
    print(f"CHECKPOINT: {ckpt.name}")
    print(f"{'=' * 60}")

    config_file = ckpt / "config.json"

    if not config_file.exists():
        print("❌ Missing config.json")
        continue

    with config_file.open(encoding="utf-8") as file:
        config = json.load(file)

    keys = [
        "model_type",
        "architectures",
        "decoder_start_token_id",
        "bos_token_id",
        "eos_token_id",
        "pad_token_id",
        "vocab_size",
    ]

    for key in keys:
        print(f"{key}: {config.get(key)}")



CHECKPOINT: checkpoint-348600
model_type: t5
architectures: ['T5ForConditionalGeneration']
decoder_start_token_id: 0
bos_token_id: 1
eos_token_id: 2
pad_token_id: 0
vocab_size: 32100

CHECKPOINT: checkpoint-406700
model_type: t5
architectures: ['T5ForConditionalGeneration']
decoder_start_token_id: 0
bos_token_id: 1
eos_token_id: 2
pad_token_id: 0
vocab_size: 32100

CHECKPOINT: checkpoint-464800
model_type: t5
architectures: ['T5ForConditionalGeneration']
decoder_start_token_id: 0
bos_token_id: 1
eos_token_id: 2
pad_token_id: 0
vocab_size: 32100

CHECKPOINT: checkpoint-522900
model_type: t5
architectures: ['T5ForConditionalGeneration']
decoder_start_token_id: 0
bos_token_id: 1
eos_token_id: 2
pad_token_id: 0
vocab_size: 32100

CHECKPOINT: checkpoint-581008
model_type: t5
architectures: ['T5ForConditionalGeneration']
decoder_start_token_id: 0
bos_token_id: 1
eos_token_id: 2
pad_token_id: 0
vocab_size: 32100


In [32]:
# ============================================================
# CELL 40 — CHECK TOKENIZER + MODEL
# ============================================================
# Mục đích:
# - Load checkpoint-348600.
# - Kiểm tra tokenizer.
# - Kiểm tra generation config của model.
# - Xác nhận decoder_start_token_id = 0 thực sự được
#   load vào model.
#
# Lưu ý:
# - Không sửa checkpoint.
# - Không chạy inference dataset.
# - Chỉ kiểm tra model trước khi smoke test.
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)

# ------------------------------------------------------------
# 1. Chọn checkpoint dùng để smoke test đầu tiên
# ------------------------------------------------------------

CKPT = (
    MODEL_DIR
    / "checkpoint-348600"
)

print("Loading checkpoint:")
print(CKPT)


# ------------------------------------------------------------
# 2. Load tokenizer
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    str(CKPT)
)


# ------------------------------------------------------------
# 3. Load T5 model
# ------------------------------------------------------------

model = AutoModelForSeq2SeqLM.from_pretrained(
    str(CKPT)
)


# ------------------------------------------------------------
# 4. Kiểm tra tokenizer
# ------------------------------------------------------------

print("\n========== TOKENIZER ==========")

print(
    "pad_token    =",
    tokenizer.pad_token
)

print(
    "pad_token_id =",
    tokenizer.pad_token_id
)

print(
    "bos_token    =",
    tokenizer.bos_token
)

print(
    "bos_token_id =",
    tokenizer.bos_token_id
)

print(
    "eos_token    =",
    tokenizer.eos_token
)

print(
    "eos_token_id =",
    tokenizer.eos_token_id
)


# ------------------------------------------------------------
# 5. Kiểm tra model.config
# ------------------------------------------------------------

print("\n========== MODEL CONFIG ==========")

print(
    "model_type =",
    model.config.model_type
)

print(
    "architectures =",
    model.config.architectures
)

print(
    "decoder_start_token_id =",
    model.config.decoder_start_token_id
)

print(
    "bos_token_id =",
    model.config.bos_token_id
)

print(
    "eos_token_id =",
    model.config.eos_token_id
)

print(
    "pad_token_id =",
    model.config.pad_token_id
)


# ------------------------------------------------------------
# 6. Kiểm tra generation_config
#
# Một số phiên bản Transformers lấy các giá trị generation
# từ model.generation_config thay vì trực tiếp từ model.config.
# Đây có thể là nguyên nhân gây lỗi trước đó.
# ------------------------------------------------------------

print("\n========== GENERATION CONFIG ==========")

print(
    "decoder_start_token_id =",
    model.generation_config.decoder_start_token_id
)

print(
    "bos_token_id =",
    model.generation_config.bos_token_id
)

print(
    "eos_token_id =",
    model.generation_config.eos_token_id
)

print(
    "pad_token_id =",
    model.generation_config.pad_token_id
)


Loading checkpoint:
/kaggle/working/MultiMend28/models/multimend-codet5-small/checkpoint-348600

========== TOKENIZER ==========
pad_token    = <pad>
pad_token_id = 0
bos_token    = <s>
bos_token_id = 1
eos_token    = </s>
eos_token_id = 2

========== MODEL CONFIG ==========
model_type = t5
architectures = ['T5ForConditionalGeneration']
decoder_start_token_id = 0
bos_token_id = 1
eos_token_id = 2
pad_token_id = 0

========== GENERATION CONFIG ==========
decoder_start_token_id = None
bos_token_id = None
eos_token_id = None
pad_token_id = None


In [33]:
# ============================================================
# CELL 41 — FIX / SYNC GENERATION CONFIG
# ============================================================
# Mục đích:
# - Không thay đổi file checkpoint trên disk.
# - Đồng bộ các token ID từ model.config sang
#   model.generation_config.
# - Khắc phục lỗi:
#   "decoder_start_token_id or bos_token_id has to be defined"
#
# Nguyên nhân đã xác định:
#   model.config.decoder_start_token_id = 0
#   model.generation_config.decoder_start_token_id = None
#
# Vì transformers.generate() sử dụng generation_config,
# cần đồng bộ hai config trước khi generate().
# ============================================================

# ------------------------------------------------------------
# Đồng bộ decoder_start_token_id
# ------------------------------------------------------------

model.generation_config.decoder_start_token_id = (
    model.config.decoder_start_token_id
)


# ------------------------------------------------------------
# Đồng bộ các token ID còn lại
# ------------------------------------------------------------

model.generation_config.bos_token_id = (
    model.config.bos_token_id
)

model.generation_config.eos_token_id = (
    model.config.eos_token_id
)

model.generation_config.pad_token_id = (
    model.config.pad_token_id
)


# ------------------------------------------------------------
# In lại để xác nhận
# ------------------------------------------------------------

print("========== GENERATION CONFIG AFTER SYNC ==========")

print(
    "decoder_start_token_id =",
    model.generation_config.decoder_start_token_id
)

print(
    "bos_token_id =",
    model.generation_config.bos_token_id
)

print(
    "eos_token_id =",
    model.generation_config.eos_token_id
)

print(
    "pad_token_id =",
    model.generation_config.pad_token_id
)


========== GENERATION CONFIG AFTER SYNC ==========
decoder_start_token_id = 0
bos_token_id = 1
eos_token_id = 2
pad_token_id = 0


In [34]:
# ============================================================
# CELL 42 — SMOKE TEST MODEL.GENERATE()
# ============================================================
# Mục đích:
# - Kiểm tra trực tiếp model.generate().
# - Chưa chạy QuixBugs.
# - Chưa chạy cả 5 checkpoint.
# - Nếu cell này PASS thì nguyên nhân lỗi generation đã được
#   xử lý.
# ============================================================

import torch

# ------------------------------------------------------------
# Chọn GPU nếu Kaggle có CUDA, ngược lại dùng CPU
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)


# ------------------------------------------------------------
# Đưa model sang device và chuyển sang evaluation mode
# ------------------------------------------------------------

model = model.to(device)
model.eval()


# ------------------------------------------------------------
# Input đơn giản để kiểm tra CodeT5
# ------------------------------------------------------------

test_input = (
    "fix bug: def add(a, b): return a - b"
)

print("\nInput:")
print(test_input)


# ------------------------------------------------------------
# Tokenize input
# ------------------------------------------------------------

inputs = tokenizer(
    test_input,
    return_tensors="pt",
    truncation=True,
    max_length=512,
)


# ------------------------------------------------------------
# Đưa tensors lên cùng device với model
# ------------------------------------------------------------

inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}


# ------------------------------------------------------------
# Generate output
# ------------------------------------------------------------

with torch.no_grad():

    outputs = model.generate(
        **inputs,

        # GenerationConfig đã được đồng bộ ở Cell 41
        max_length=128,

        # Baseline deterministic
        do_sample=False,
    )


# ------------------------------------------------------------
# Decode prediction
# ------------------------------------------------------------

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
)


# ------------------------------------------------------------
# In kết quả
# ------------------------------------------------------------

print("\nGenerated output:")
print(generated_text)

print("\n✅ model.generate() smoke test PASSED")


Device: cuda

Input:
fix bug: def add(a, b): return a - b

Generated output:
fix bug: http://www.w3.org/1999/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/2001/

✅ model.generate() smoke test PASSED


In [36]:
# ============================================================
# CELL 43 — INSPECT QUIXBUGS INPUT FORMAT
# ============================================================
# Mục đích:
# - Không tự đoán format input của model.
# - Kiểm tra chính xác dữ liệu QuixBugs đang được pipeline
#   sử dụng.
# - Xác định input column và cách preprocessing.
#
# QUAN TRỌNG:
# - Chưa chạy model.
# - Chỉ đọc dữ liệu và in một vài sample.
# ============================================================

from pathlib import Path
import json

REPO = Path(
    "/kaggle/working/MultiMend28"
)

print("Repository:", REPO)

# ------------------------------------------------------------
# Tìm các file dữ liệu có khả năng là QuixBugs
# ------------------------------------------------------------

candidates = []

for pattern in [
    "*.json",
    "*.jsonl",
    "*.arrow",
    "*.parquet",
    "*.csv",
]:

    for path in REPO.rglob(pattern):

        name = path.name.lower()

        if (
            "quix" in name
            or "test" in name
            or "bug" in name
        ):
            candidates.append(path)

print("\nPossible dataset files:")

for path in candidates[:50]:
    print(path)


Repository: /kaggle/working/MultiMend28

Possible dataset files:
/kaggle/working/MultiMend28/outputs/QuixBugs-Python/QuixBugs_Python.jsonl


In [37]:
# ============================================================
# CELL 44 — INSPECT INFERENCE SCRIPT
# ============================================================
# Mục đích:
# - Đọc generate_primary_candidates.py.
# - Tìm:
#     + cách load dataset
#     + preprocessing/tokenization
#     + input column
#     + prompt/template
#     + model.generate()
# - Không sửa file.
# ============================================================

from pathlib import Path

script_path = (
    Path("/kaggle/working/MultiMend28")
    / "src"
    / "generate_primary_candidates.py"
)

print("Script:")
print(script_path)

if not script_path.exists():
    raise FileNotFoundError(script_path)

text = script_path.read_text(
    encoding="utf-8"
)

# ------------------------------------------------------------
# In toàn bộ script để kiểm tra pipeline.
# Nếu script dài, có thể xem phần liên quan generate_candidates
# trước.
# ------------------------------------------------------------

print(text)


Script:
/kaggle/working/MultiMend28/src/generate_primary_candidates.py
import json
import string
from collections import ChainMap, defaultdict
from itertools import chain
from pathlib import Path

import torch
from datasets import Dataset, concatenate_datasets
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    set_seed,
)

from .configs import (
    bugaid_gen_dir,
    bugsinpy_gen_dir,
    codeflaws_gen_dir,
    d4j_gen_dir,
    models_root,
    quixbugs_genjava_dir,
    quixbugs_genpy_dir,
    runbugrunjs_gen_dir,
)
from .rag_utils import RAG

set_seed(42)

# Config
dataset = "QuixBugs-Python"
model_name = "multimend-codet5-small"


def get_checkpoints(checkpoints_dir: Path) -> list[tuple[str, Path]]:
    """Get the list of checkpoints from the checkpoints directory"""

    checkpoints: dict[str, Path] = {}
    for d in checkpoints_dir.iterdir():
        name_parts = d.name.split("-")
        if (
            len(name_parts) == 2
            and name_parts[0

In [38]:
# ============================================================
# CELL 45 — INSPECT INFERENCE / GENERATION PIPELINE
# ============================================================
# Mục đích:
# - Kiểm tra chính xác cách project:
#     + chọn dataset QuixBugs-Python
#     + tạo input
#     + load checkpoint
#     + gọi model.generate()
#     + ghi output
#
# - Không chạy inference.
# - Không sửa source code.
#
# Đây là bước cần thiết trước khi sửa generate_primary_candidates.py
# vì ta phải giữ nguyên preprocessing của project.
# ============================================================

from pathlib import Path

script_path = (
    Path("/kaggle/working/MultiMend28")
    / "src"
    / "generate_primary_candidates.py"
)

lines = script_path.read_text(
    encoding="utf-8"
).splitlines()

# ------------------------------------------------------------
# In các dòng từ phần tokenize/generation trở đi.
# ------------------------------------------------------------

for line_number, line in enumerate(
    lines[80:260],
    start=81
):
    print(
        f"{line_number:4d}: {line}"
    )


  81: 
  82:             print(bugid, h)
  83:             if rag_result:
  84:                 test_input = f"{source} {rag_result} {context}".replace(
  85:                     tokenizer.eos_token, tokenizer.unk_token
  86:                 )
  87:             else:
  88:                 test_input = f"{source} {context}".replace(
  89:                     tokenizer.eos_token, tokenizer.unk_token
  90:                 )
  91:             test_data["inputs"].append(test_input)
  92: 
  93:     test_dataset = Dataset.from_dict(test_data)
  94:     return test_dataset
  95: 
  96: 
  97: def tokenize_data(examples):
  98:     inputs = tokenizer(
  99:         examples["inputs"],
 100:         padding="longest",
 101:         truncation=True,
 102:         max_length=max_input_length,
 103:     )
 104:     return inputs
 105: 
 106: 
 107: def generate_candidates(examples, model, ch_name):
 108:     input_ids = examples["input_ids"].to(device)
 109:     attention_mask = examples["attentio

In [39]:
# ============================================================
# CELL 46 — PATCH PRIMARY INFERENCE SCRIPT
# ============================================================
# Mục đích:
# 1. Backup generate_primary_candidates.py trước khi sửa.
#
# 2. Sửa lỗi GenerationConfig:
#       model.config.decoder_start_token_id = 0
#       nhưng
#       model.generation_config.decoder_start_token_id = None
#
# 3. Đảm bảo 5 checkpoint cuối được chọn dựa trên
#    trainer_state.json -> global_step,
#    KHÔNG chỉ dựa vào tên thư mục.
#
# 4. Không thay đổi checkpoint trên disk.
# ============================================================

from pathlib import Path
import shutil

REPO = Path(
    "/kaggle/working/MultiMend28"
)

SCRIPT = (
    REPO
    / "src"
    / "generate_primary_candidates.py"
)

BACKUP = (
    REPO
    / "src"
    / "generate_primary_candidates.py.bak"
)


# ------------------------------------------------------------
# 1. Kiểm tra source script
# ------------------------------------------------------------

if not SCRIPT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy: {SCRIPT}"
    )


# ------------------------------------------------------------
# 2. Backup script gốc
# ------------------------------------------------------------

if not BACKUP.exists():
    shutil.copy2(
        SCRIPT,
        BACKUP
    )

    print(
        "Đã backup script gốc:"
    )
    print(BACKUP)

else:
    print(
        "Backup đã tồn tại:"
    )
    print(BACKUP)


# ------------------------------------------------------------
# 3. Đọc source code
# ------------------------------------------------------------

text = SCRIPT.read_text(
    encoding="utf-8"
)


# ============================================================
# PATCH A — chọn checkpoint theo trainer_state.json
# ============================================================

old_get_checkpoints = '''def get_checkpoints(checkpoints_dir: Path) -> list[tuple[str, Path]]:
    """Get the list of checkpoints from the checkpoints directory"""

    checkpoints: dict[str, Path] = {}
    for d in checkpoints_dir.iterdir():
        name_parts = d.name.split("-")
        if (
            len(name_parts) == 2
            and name_parts[0] == "checkpoint"
            and name_parts[1].isdigit()
        ):
            checkpoints[d.name] = d

    sorted_checkpoints = sorted(
        checkpoints.items(), key=lambda x: int(x[0].split("-")[1])
    )

    return sorted_checkpoints
'''

new_get_checkpoints = '''def get_checkpoints(checkpoints_dir: Path) -> list[tuple[str, Path]]:
    """Get checkpoints sorted by trainer_state.json global_step."""

    checkpoints = []

    for d in checkpoints_dir.iterdir():
        name_parts = d.name.split("-")

        if not (
            len(name_parts) == 2
            and name_parts[0] == "checkpoint"
            and name_parts[1].isdigit()
        ):
            continue

        state_file = d / "trainer_state.json"

        if not state_file.exists():
            print(f"WARNING: missing trainer_state.json: {d}")
            continue

        with state_file.open(encoding="utf-8") as file:
            state = json.load(file)

        global_step = state.get("global_step")

        if global_step is None:
            print(f"WARNING: missing global_step: {d}")
            continue

        checkpoints.append(
            (d.name, d, int(global_step))
        )

    checkpoints.sort(key=lambda x: x[2])

    return [
        (name, path)
        for name, path, _ in checkpoints
    ]
'''

if old_get_checkpoints not in text:
    raise RuntimeError(
        "Không tìm thấy hàm get_checkpoints() "
        "theo đúng nội dung dự kiến."
    )

text = text.replace(
    old_get_checkpoints,
    new_get_checkpoints
)


# ============================================================
# PATCH B — sync GenerationConfig sau khi load model
# ============================================================

old_model_load = '''    model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(device)
    results = tokenized_test_dataset.map(
'''

new_model_load = '''    model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint).to(device)

    # --------------------------------------------------------
    # Sync GenerationConfig với model.config.
    #
    # Một số checkpoint có decoder_start_token_id trong
    # model.config nhưng generation_config lại để None.
    # transformers.generate() sử dụng generation_config,
    # nên cần đồng bộ trước khi generate().
    # --------------------------------------------------------

    model.generation_config.decoder_start_token_id = (
        model.config.decoder_start_token_id
    )

    model.generation_config.bos_token_id = (
        model.config.bos_token_id
    )

    model.generation_config.eos_token_id = (
        model.config.eos_token_id
    )

    model.generation_config.pad_token_id = (
        model.config.pad_token_id
    )

    if model.generation_config.decoder_start_token_id is None:
        raise ValueError(
            f"Checkpoint {ch_name} has no "
            "decoder_start_token_id."
        )

    print(
        "decoder_start_token_id =",
        model.generation_config.decoder_start_token_id
    )

    results = tokenized_test_dataset.map(
'''

if old_model_load not in text:
    raise RuntimeError(
        "Không tìm thấy đoạn load model "
        "theo đúng nội dung dự kiến."
    )

text = text.replace(
    old_model_load,
    new_model_load
)


# ------------------------------------------------------------
# 4. Ghi source code đã sửa
# ------------------------------------------------------------

SCRIPT.write_text(
    text,
    encoding="utf-8"
)

print("\nĐã patch:")
print(SCRIPT)


Đã backup script gốc:
/kaggle/working/MultiMend28/src/generate_primary_candidates.py.bak

Đã patch:
/kaggle/working/MultiMend28/src/generate_primary_candidates.py


In [40]:
# ============================================================
# CELL 47 — VERIFY INFERENCE SCRIPT PATCH
# ============================================================
# Mục đích:
# - Xác nhận get_checkpoints() đã đọc trainer_state.json.
# - Xác nhận GenerationConfig được sync.
# - Không chạy model inference.
# ============================================================

from pathlib import Path

SCRIPT = Path(
    "/kaggle/working/MultiMend28/src/"
    "generate_primary_candidates.py"
)

text = SCRIPT.read_text(
    encoding="utf-8"
)


# ------------------------------------------------------------
# Kiểm tra PATCH A
# ------------------------------------------------------------

assert (
    'state_file = d / "trainer_state.json"'
    in text
)

assert (
    'global_step = state.get("global_step")'
    in text
)


# ------------------------------------------------------------
# Kiểm tra PATCH B
# ------------------------------------------------------------

assert (
    "model.generation_config.decoder_start_token_id"
    in text
)

assert (
    "model.config.decoder_start_token_id"
    in text
)


print(
    "✅ Checkpoint selection patch: OK"
)

print(
    "✅ GenerationConfig patch: OK"
)

print(
    "✅ Source verification PASSED"
)


✅ Checkpoint selection patch: OK
✅ GenerationConfig patch: OK
✅ Source verification PASSED


In [41]:
# ============================================================
# CELL 48 — QUIXBUGS-PYTHON SMOKE TEST
# ============================================================
# Mục đích:
# - Chạy pipeline inference thật của project.
# - Dataset: QuixBugs-Python.
# - Seed trong script: 42.
# - Sử dụng 5 checkpoint cuối theo trainer_state.json.
# - Giữ nguyên beam_size = 100.
# - Giữ nguyên num_return_sequences = 100.
#
# Lưu ý:
# - Pipeline hiện tại sẽ chạy 5 checkpoint.
# - Với 40 mẫu và 100 candidates/checkpoint,
#   số lượng output sẽ khá lớn.
# ============================================================

import subprocess
from pathlib import Path

REPO = Path(
    "/kaggle/working/MultiMend28"
)

command = [
    "/usr/bin/python3",
    "-m",
    "src.generate_primary_candidates",
]

print(
    "$",
    " ".join(command)
)

subprocess.run(
    command,
    cwd=REPO,
    check=True,
)

print(
    "\n✅ QuixBugs-Python inference completed."
)


$ /usr/bin/python3 -m src.generate_primary_candidates


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


# RAG retrievals...
bitcount 0
breadth_first_search 0
bucketsort 0
depth_first_search 0
detect_cycle 0
find_first_in_sorted 0
find_in_sorted 0
flatten 0
gcd 0
get_factors 0
hanoi 0
is_valid_parenthesization 0
kheapsort 0
knapsack 0
kth 0
lcs_length 0
levenshtein 0
lis 0
longest_common_subsequence 0
max_sublist_sum 0
mergesort 0
minimum_spanning_tree 0
next_palindrome 0
next_permutation 0
pascal 0
possible_change 0
powerset 0
quicksort 0
reverse_linked_list 0
rpn_eval 0
shortest_path_length 0
shortest_path_lengths 0
shortest_paths 0
shunting_yard 0
sieve 0
sqrt 0
subsequences 0
to_base 0
topological_ordering 0
wrap 0
Tokenizing...


Map: 100%|██████████| 40/40 [00:00<00:00, 2676.22 examples/s]


Generating from checkpoint-348600...
decoder_start_token_id = 0


...

Generating from checkpoint-522900...
decoder_start_token_id = 0


Map: 100%|██████████| 40/40 [00:24<00:00,  1.64 examples/s]


Generating from checkpoint-581008...
decoder_start_token_id = 0


Creating json from Arrow format: 100%|██████████| 20/20 [00:00<00:00, 368.25ba/s]



✅ QuixBugs-Python inference completed.


In [42]:
# ============================================================
# CELL 49 — VERIFY QUIXBUGS JSONL OUTPUT
# ============================================================
# Mục đích:
# - Tìm file JSONL được sinh bởi inference.
# - Kiểm tra JSONL có đọc được hay không.
# - Kiểm tra các field chính:
#       checkpoint
#       decoded_sequences
#       sequences_scores
#       bugid
#       hunk
# - Đếm số dòng output.
#
# Với:
#   40 mẫu
#   5 checkpoint
#   100 candidates/checkpoint
#
# kỳ vọng khoảng:
#   40 × 5 = 200 records
#
# Mỗi record chứa 100 decoded_sequences.
# ============================================================

import json
from pathlib import Path

REPO = Path(
    "/kaggle/working/MultiMend28"
)

# ------------------------------------------------------------
# Thư mục output của QuixBugs-Python
# ------------------------------------------------------------

output_root = REPO

jsonl_files = sorted(
    output_root.rglob("sequences_*.jsonl")
)

print("Found JSONL files:")

for path in jsonl_files:
    print(" -", path)


# ------------------------------------------------------------
# Kiểm tra từng JSONL
# ------------------------------------------------------------

for path in jsonl_files:

    print("\n" + "=" * 70)
    print("FILE:", path)
    print("=" * 70)

    valid_lines = 0
    invalid_lines = 0

    checkpoints = set()
    bugids = set()

    first_record = None

    with path.open(
        "r",
        encoding="utf-8"
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1
        ):

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

            except json.JSONDecodeError as exc:

                invalid_lines += 1

                print(
                    f"❌ Invalid JSON at line "
                    f"{line_number}: {exc}"
                )

                continue

            valid_lines += 1

            # Lưu record đầu tiên để inspect schema
            if first_record is None:
                first_record = record

            # Thu thập checkpoint
            if "checkpoint" in record:

                checkpoints.update(
                    record["checkpoint"]
                )

            # Thu thập bug IDs
            if "bugid" in record:

                bugids.update(
                    record["bugid"]
                )


    print("\nValid JSON lines  :", valid_lines)
    print("Invalid JSON lines:", invalid_lines)

    print(
        "\nCheckpoints found:"
    )

    for checkpoint in sorted(checkpoints):
        print(" -", checkpoint)

    print(
        "\nNumber of unique bugids:",
        len(bugids)
    )


    # --------------------------------------------------------
    # Inspect record đầu tiên
    # --------------------------------------------------------

    if first_record is not None:

        print(
            "\nFirst record keys:"
        )

        print(
            list(first_record.keys())
        )

        print(
            "\nFirst record:"
        )

        print(
            json.dumps(
                first_record,
                indent=2,
                ensure_ascii=False
            )[:5000]
        )


Found JSONL files:
 - /kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend/sequences_100.jsonl

FILE: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend/sequences_100.jsonl

Valid JSON lines  : 20000
Invalid JSON lines: 0

Checkpoints found:
 - -
 - 0
 - 1
 - 2
 - 3
 - 4
 - 5
 - 6
 - 7
 - 8
 - 9
 - c
 - e
 - h
 - i
 - k
 - n
 - o
 - p
 - t

Number of unique bugids: 26

First record keys:
['checkpoint', 'decoded_sequences', 'sequences_scores', 'bugid', 'hunk']

First record:
{
  "checkpoint": "checkpoint-348600",
  "decoded_sequences": "n ^= n - 1",
  "sequences_scores": -0.2513872981,
  "bugid": "bitcount",
  "hunk": 0
}


In [43]:
# ============================================================
# CELL 50 — VALIDATE BASELINE OUTPUT SCHEMA
# ============================================================
# Mục đích:
# - Kiểm tra chính xác 20,000 candidate.
# - Xác nhận đủ 5 checkpoint.
# - Xác nhận mỗi checkpoint có 40 × 100 candidates.
# - Xác nhận đủ 40 bug/hunk.
# - Kiểm tra JSONL không bị lỗi.
#
# Kỳ vọng:
#
#   40 bugs
#   × 5 checkpoints
#   × 100 candidates
#   = 20,000 JSONL records
#
# Không thực hiện ensemble.
# Không thực hiện patch ranking.
# ============================================================

import json
from collections import Counter
from pathlib import Path


# ------------------------------------------------------------
# 1. Xác định output JSONL
# ------------------------------------------------------------

OUTPUT = Path(
    "/kaggle/working/MultiMend28/"
    "outputs/QuixBugs-Python/"
    "outputs-multimend/"
    "sequences_100.jsonl"
)

if not OUTPUT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy output: {OUTPUT}"
    )


# ------------------------------------------------------------
# 2. Đọc JSONL
# ------------------------------------------------------------

records = []

with OUTPUT.open(
    "r",
    encoding="utf-8"
) as file:

    for line_number, line in enumerate(
        file,
        start=1
    ):

        line = line.strip()

        if not line:
            continue

        record = json.loads(line)

        records.append(record)


# ------------------------------------------------------------
# 3. Tổng số candidate
# ------------------------------------------------------------

print("Total JSONL records:", len(records))


# ------------------------------------------------------------
# 4. Kiểm tra schema
# ------------------------------------------------------------

required_keys = {
    "checkpoint",
    "decoded_sequences",
    "sequences_scores",
    "bugid",
    "hunk",
}

for i, record in enumerate(records):

    missing = (
        required_keys
        - set(record.keys())
    )

    if missing:

        raise ValueError(
            f"Record {i} thiếu field: {missing}"
        )


print(
    "Schema validation: PASS"
)


# ------------------------------------------------------------
# 5. Đếm checkpoint
# ------------------------------------------------------------

checkpoint_counts = Counter(
    record["checkpoint"]
    for record in records
)

print("\nCheckpoint counts:")

for checkpoint, count in sorted(
    checkpoint_counts.items()
):

    print(
        f"  {checkpoint}: {count}"
    )


# ------------------------------------------------------------
# 6. Đếm bug
# ------------------------------------------------------------

bug_counts = Counter(
    record["bugid"]
    for record in records
)

print(
    "\nUnique bugids:",
    len(bug_counts)
)

print(
    "Bug/hunk combinations:",
    len({
        (
            record["bugid"],
            record["hunk"]
        )
        for record in records
    })
)


# ------------------------------------------------------------
# 7. Kiểm tra 5 checkpoint
# ------------------------------------------------------------

expected_checkpoints = {
    "checkpoint-348600",
    "checkpoint-406700",
    "checkpoint-464800",
    "checkpoint-522900",
    "checkpoint-581008",
}

actual_checkpoints = set(
    checkpoint_counts.keys()
)

assert actual_checkpoints == expected_checkpoints, (
    "Checkpoint không đúng.\n"
    f"Expected: {expected_checkpoints}\n"
    f"Actual:   {actual_checkpoints}"
)


# ------------------------------------------------------------
# 8. Mỗi checkpoint phải có 4,000 candidates
#
# 40 inputs × 100 candidates = 4,000
# ------------------------------------------------------------

for checkpoint in expected_checkpoints:

    assert (
        checkpoint_counts[checkpoint] == 4000
    ), (
        f"{checkpoint} có "
        f"{checkpoint_counts[checkpoint]} records, "
        f"không phải 4000."
    )


# ------------------------------------------------------------
# 9. Tổng số candidate
# ------------------------------------------------------------

assert len(records) == 20_000, (
    f"Expected 20,000 records, "
    f"got {len(records)}"
)


# ------------------------------------------------------------
# 10. Kiểm tra score có tồn tại
# ------------------------------------------------------------

missing_scores = sum(
    record["sequences_scores"] is None
    for record in records
)

print(
    "\nMissing sequence scores:",
    missing_scores
)

assert missing_scores == 0


# ------------------------------------------------------------
# 11. In một vài candidate mẫu
# ------------------------------------------------------------

print("\nSample candidates:")

for record in records[:10]:

    print(
        record["checkpoint"],
        "|",
        record["bugid"],
        "| hunk =",
        record["hunk"],
        "| score =",
        record["sequences_scores"],
        "|",
        repr(record["decoded_sequences"])
    )


# ------------------------------------------------------------
# Kết luận
# ------------------------------------------------------------

print(
    "\n" + "=" * 60
)

print(
    "✅ BASELINE JSONL VALIDATION PASSED"
)

print(
    "20,000 candidates"
)

print(
    "5 checkpoints × 40 inputs × 100 candidates"
)

print(
    "No checkpoint ensemble performed."
)

print(
    "=" * 60
)


Total JSONL records: 20000
Schema validation: PASS

Checkpoint counts:
  checkpoint-348600: 4000
  checkpoint-406700: 4000
  checkpoint-464800: 4000
  checkpoint-522900: 4000
  checkpoint-581008: 4000

Unique bugids: 40
Bug/hunk combinations: 40

Missing sequence scores: 0

Sample candidates:
checkpoint-348600 | bitcount | hunk = 0 | score = -0.2513872981 | 'n ^= n - 1'
checkpoint-348600 | bitcount | hunk = 0 | score = -0.3517662287 | 'n ^= (n - 1)'
checkpoint-348600 | bitcount | hunk = 0 | score = -0.4465347826 | 'n ^= n - 1.0'
checkpoint-348600 | bitcount | hunk = 0 | score = -0.570260644 | 'n ^= int(n - 1)'
checkpoint-348600 | bitcount | hunk = 0 | score = -0.5920192003 | 'n ^= n'
checkpoint-348600 | bitcount | hunk = 0 | score = -0.5946378708 | 'n &= n - 1'
checkpoint-348600 | bitcount | hunk = 0 | score = -0.5967445374 | 'if n: n ^= n - 1'
checkpoint-348600 | bitcount | hunk = 0 | score = -0.5993896127 | 'n ^= n - 1 - 1'
checkpoint-348600 | bitcount | hunk = 0 | score = -0.6030863

In [44]:
# ============================================================
# CELL 51 — BASELINE MANIFEST / CONFIG HASH
# ============================================================
# Mục đích:
# - Ghi lại đầy đủ cấu hình của baseline inference.
# - Seed cố định = 42.
# - Ghi chính xác 5 checkpoint được sử dụng.
# - Xác nhận global_step từ trainer_state.json.
# - Tạo SHA256 hash cho configuration.
# - Ghi input/output JSONL.
#
# Lưu ý:
# - Không chạy inference.
# - Không ensemble.
# - Không thay đổi checkpoint.
# ============================================================

import hashlib
import json
from pathlib import Path


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

REPO = Path(
    "/kaggle/working/MultiMend28"
)

MODEL_DIR = (
    REPO
    / "models"
    / "multimend-codet5-small"
)

OUTPUT_DIR = (
    REPO
    / "outputs"
    / "QuixBugs-Python"
    / "outputs-multimend"
)

INPUT_JSONL = (
    OUTPUT_DIR
    / "generated_input.jsonl"
)

OUTPUT_JSONL = (
    OUTPUT_DIR
    / "sequences_100.jsonl"
)


# ------------------------------------------------------------
# 2. Baseline configuration
# ------------------------------------------------------------

config = {
    "dataset": "QuixBugs-Python",
    "model_name": "multimend-codet5-small",

    "seed": 42,

    "max_input_length": 512,
    "max_target_length": 256,
    "min_target_length": 0,

    "beam_size": 100,
    "num_return_sequences": 100,
    "batch_size": 1,
    "num_checkpoints": 5,

    "device": (
        "cuda"
        if __import__("torch").cuda.is_available()
        else "cpu"
    ),

    "input_jsonl": str(INPUT_JSONL),
    "output_jsonl": str(OUTPUT_JSONL),
}


# ------------------------------------------------------------
# 3. Đọc 5 checkpoint và trainer_state.json
# ------------------------------------------------------------

checkpoint_info = []

for checkpoint_dir in sorted(
    MODEL_DIR.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[1])
):

    state_file = (
        checkpoint_dir
        / "trainer_state.json"
    )

    if not state_file.exists():
        continue

    with state_file.open(
        encoding="utf-8"
    ) as file:

        state = json.load(file)

    global_step = state.get(
        "global_step"
    )

    checkpoint_info.append(
        {
            "name": checkpoint_dir.name,
            "path": str(checkpoint_dir),
            "global_step": global_step,
        }
    )


# ------------------------------------------------------------
# 4. Chọn 5 checkpoint cuối theo global_step
# ------------------------------------------------------------

checkpoint_info.sort(
    key=lambda x: x["global_step"]
)

selected_checkpoints = (
    checkpoint_info[-5:]
)

config["checkpoints"] = [
    {
        "name": item["name"],
        "global_step": item["global_step"],
    }
    for item in selected_checkpoints
]


# ------------------------------------------------------------
# 5. Tạo canonical JSON
#
# sort_keys=True giúp hash ổn định.
# ------------------------------------------------------------

canonical_config = json.dumps(
    config,
    sort_keys=True,
    separators=(",", ":"),
    ensure_ascii=False,
)


# ------------------------------------------------------------
# 6. SHA256 config hash
# ------------------------------------------------------------

config_hash = hashlib.sha256(
    canonical_config.encode("utf-8")
).hexdigest()

config["config_hash"] = (
    f"sha256:{config_hash}"
)


# ------------------------------------------------------------
# 7. Tạo manifest
# ------------------------------------------------------------

manifest = {
    "experiment": "MultiMend28 baseline inference",

    "config": config,

    "selected_checkpoints": selected_checkpoints,

    "input": {
        "path": str(INPUT_JSONL),
        "exists": INPUT_JSONL.exists(),
    },

    "output": {
        "path": str(OUTPUT_JSONL),
        "exists": OUTPUT_JSONL.exists(),
    },
}


# ------------------------------------------------------------
# 8. Lưu manifest
# ------------------------------------------------------------

MANIFEST = (
    OUTPUT_DIR
    / "baseline_manifest.json"
)

with MANIFEST.open(
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        manifest,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 9. In thông tin quan trọng
# ------------------------------------------------------------

print("=" * 70)
print("BASELINE MANIFEST")
print("=" * 70)

print(
    "\nDataset:",
    config["dataset"]
)

print(
    "Model:",
    config["model_name"]
)

print(
    "Seed:",
    config["seed"]
)

print(
    "Config hash:",
    config["config_hash"]
)

print(
    "\nSelected checkpoints:"
)

for checkpoint in selected_checkpoints:

    print(
        f"  {checkpoint['name']} "
        f"(global_step={checkpoint['global_step']})"
    )

print(
    "\nInput JSONL:",
    INPUT_JSONL
)

print(
    "Output JSONL:",
    OUTPUT_JSONL
)

print(
    "\nManifest:",
    MANIFEST
)

print(
    "\n✅ Baseline manifest created."
)


BASELINE MANIFEST

Dataset: QuixBugs-Python
Model: multimend-codet5-small
Seed: 42
Config hash: sha256:c0c516a4f019eabf3b648e39a901e7b81571636ba97dbc31ce8d64063962eaaa

Selected checkpoints:
  checkpoint-348600 (global_step=348600)
  checkpoint-406700 (global_step=406700)
  checkpoint-464800 (global_step=464800)
  checkpoint-522900 (global_step=522900)
  checkpoint-581008 (global_step=581008)

Input JSONL: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend/generated_input.jsonl
Output JSONL: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend/sequences_100.jsonl

Manifest: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend/baseline_manifest.json

✅ Baseline manifest created.


In [46]:
# ============================================================
# CELL 52 — INSPECT OFFICIAL QUIXBUGS-PYTHON VALIDATOR
# ============================================================
# Mục đích:
# - Đọc validator chính thức của MultiMend28.
# - Xác định:
#     + input arguments
#     + output arguments
#     + cách apply candidate patch
#     + cách chạy test
#     + cách ghi kết quả
#
# - Không chạy evaluation ở cell này.
# ============================================================

from pathlib import Path

REPO = Path(
    "/kaggle/working/MultiMend28"
)

VALIDATOR = (
    REPO
    / "src"
    / "validators"
    / "validate_quixbugs_python.py"
)

if not VALIDATOR.exists():
    raise FileNotFoundError(
        f"Không tìm thấy validator: {VALIDATOR}"
    )

print("=" * 70)
print("OFFICIAL QUIXBUGS-PYTHON VALIDATOR")
print("=" * 70)

print(
    VALIDATOR.read_text(
        encoding="utf-8"
    )
)


OFFICIAL QUIXBUGS-PYTHON VALIDATOR
import contextlib
import json
import shutil
import subprocess
import threading
import time
import timeit
from collections import ChainMap
from copy import deepcopy
from enum import Enum, auto
from pathlib import Path
from typing import Optional

import joblib
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from tqdm import tqdm

from ..configs import quixbugs_dir, quixbugs_genpy_dir

project_dir = quixbugs_dir
gen_dir = quixbugs_genpy_dir
bugs_metadata_file = "QuixBugs_Python.jsonl"
model = "multimend"

output_dir = gen_dir / f"outputs-{model}"
temp_dir = output_dir / "temp"
save_state_dir = output_dir / "save-state"
output_size = 100

rem_file_path = gen_dir / "rem.txt"
add_file_path = gen_dir / "add.txt"

with (
    open(rem_file_path) as rem_file,
    open(add_file_path) as add_file,
):
    sources = [src.strip() for src in rem_file]
    targets = [tgt.strip() for tgt in add_file]


@contextlib.contextmanager
def tqdm_jo

In [47]:
# ============================================================
# CELL 53 — FIND FINAL CANDIDATE PIPELINE
# ============================================================
# Mục đích:
# - Tìm script tạo final_candidates_100.jsonl.
# - Xác định bước ranking/filter candidate của MultiMend28.
# - Không chạy evaluation ở cell này.
#
# Luồng chính thức dự kiến:
#
#   sequences_100.jsonl
#          ↓
#   candidate selection / ranking
#          ↓
#   final_candidates_100.jsonl
#          ↓
#   validate_quixbugs_python.py
#          ↓
#   plausible_candidates_100.jsonl
# ============================================================

from pathlib import Path

REPO = Path(
    "/kaggle/working/MultiMend28"
)

OUTPUT_DIR = (
    REPO
    / "outputs"
    / "QuixBugs-Python"
    / "outputs-multimend"
)

print("=" * 70)
print("EXISTING CANDIDATE OUTPUTS")
print("=" * 70)

for path in sorted(OUTPUT_DIR.glob("*.jsonl")):
    print(path)

print("\n" + "=" * 70)
print("SCRIPTS MENTIONING final_candidates")
print("=" * 70)

for path in REPO.rglob("*.py"):

    try:
        text = path.read_text(
            encoding="utf-8"
        )
    except Exception:
        continue

    if "final_candidates" in text:
        print(path)


EXISTING CANDIDATE OUTPUTS
/kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend/generated_input.jsonl
/kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend/sequences_100.jsonl

SCRIPTS MENTIONING final_candidates
/kaggle/working/MultiMend28/src/combine_checkpoints_results.py
/kaggle/working/MultiMend28/src/validators/validate_codeflaws.py
/kaggle/working/MultiMend28/src/validators/validate_runbugrunjs.py
/kaggle/working/MultiMend28/src/validators/validate_bugsinpy.py
/kaggle/working/MultiMend28/src/validators/validate_quixbugs_java.py
/kaggle/working/MultiMend28/src/validators/validate_defects4j.py
/kaggle/working/MultiMend28/src/validators/validate_quixbugs_python.py


In [48]:
# ============================================================
# CELL 54 — INSPECT CHECKPOINT COMBINATION SCRIPT
# ============================================================
# Mục đích:
# - Đọc script chính thức combine kết quả từ nhiều checkpoint.
# - Xác định chính xác:
#     + cách gộp 5 checkpoint
#     + cách ranking candidate
#     + cách loại duplicate
#     + cách tạo final_candidates_100.jsonl
#
# Lưu ý:
# - Chưa chạy script.
# - Chưa validation patch.
# - Chưa thay đổi sequences_100.jsonl.
# ============================================================

from pathlib import Path

REPO = Path(
    "/kaggle/working/MultiMend28"
)

SCRIPT = (
    REPO
    / "src"
    / "combine_checkpoints_results.py"
)

if not SCRIPT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy: {SCRIPT}"
    )

print("=" * 70)
print("COMBINE CHECKPOINTS SCRIPT")
print("=" * 70)

print(
    SCRIPT.read_text(
        encoding="utf-8"
    )
)


COMBINE CHECKPOINTS SCRIPT
from itertools import chain

import pandas as pd

from .configs import (
    bugaid_gen_dir,
    bugsinpy_gen_dir,
    codeflaws_gen_dir,
    d4j_gen_dir,
    quixbugs_genjava_dir,
    quixbugs_genpy_dir,
    runbugrunjs_gen_dir,
)

# Config
dataset = "QuixBugs-Python"
model = "multimend"

if dataset == "QuixBugs-Python":
    gen_dir = quixbugs_genpy_dir
    bugs_metadata_file = "QuixBugs_Python.jsonl"
elif dataset == "QuixBugs-Java":
    gen_dir = quixbugs_genjava_dir
    bugs_metadata_file = "QuixBugs_Java.jsonl"
elif dataset == "Defects4J":
    gen_dir = d4j_gen_dir
    bugs_metadata_file = "Defects4J.jsonl"
elif dataset == "BugAID":
    gen_dir = bugaid_gen_dir
    bugs_metadata_file = "BugAID.jsonl"
elif dataset == "Codeflaws":
    gen_dir = codeflaws_gen_dir
    bugs_metadata_file = "Codeflaws.jsonl"
elif dataset == "BugsInPy":
    gen_dir = bugsinpy_gen_dir
    bugs_metadata_file = "BugsInPy.jsonl"
elif dataset == "RunBugRun-JS":
    gen_dir = runbugru

In [49]:
# ============================================================
# CELL 55 — COMBINE CHECKPOINT RESULTS
# ============================================================
# Mục đích:
# - Dùng pipeline chính thức của MultiMend28.
# - Gộp candidate từ 5 checkpoint đã inference.
# - Normalize và deduplicate candidate patches.
# - Tạo exact_match flag.
#
# Input:
#   sequences_100.jsonl
#
# Output:
#   final_candidates_100.jsonl
#
# Quan trọng:
# - Không train lại model.
# - Không chạy model.generate().
# - Không ensemble xác suất/weights giữa checkpoint.
# - Không chọn checkpoint dựa trên tên thư mục.
# - 5 checkpoint đã được xác nhận bằng trainer_state.json.
# ============================================================

import subprocess
import sys
from pathlib import Path


REPO = Path(
    "/kaggle/working/MultiMend28"
)

INPUT = (
    REPO
    / "outputs"
    / "QuixBugs-Python"
    / "outputs-multimend"
    / "sequences_100.jsonl"
)

OUTPUT = (
    REPO
    / "outputs"
    / "QuixBugs-Python"
    / "outputs-multimend"
    / "final_candidates_100.jsonl"
)


# ------------------------------------------------------------
# Kiểm tra input
# ------------------------------------------------------------

if not INPUT.exists():
    raise FileNotFoundError(
        f"Không tìm thấy input: {INPUT}"
    )


# ------------------------------------------------------------
# Chạy script chính thức
# ------------------------------------------------------------

print("=" * 70)
print("Running official checkpoint combination")
print("=" * 70)

command = [
    sys.executable,
    "-m",
    "src.combine_checkpoints_results",
]

print(
    "$",
    " ".join(command)
)

subprocess.run(
    command,
    cwd=REPO,
    check=True,
)


# ------------------------------------------------------------
# Kiểm tra output
# ------------------------------------------------------------

if not OUTPUT.exists():
    raise FileNotFoundError(
        f"Không tạo được output: {OUTPUT}"
    )


print("\n" + "=" * 70)
print("COMBINATION COMPLETED")
print("=" * 70)

print(
    "Input :",
    INPUT
)

print(
    "Output:",
    OUTPUT
)

print(
    "\n✅ final_candidates_100.jsonl created."
)


Running official checkpoint combination
$ /usr/bin/python3 -m src.combine_checkpoints_results
All: 20000
Deduped: 9667

COMBINATION COMPLETED
Input : /kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend/sequences_100.jsonl
Output: /kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend/final_candidates_100.jsonl

✅ final_candidates_100.jsonl created.


In [70]:
# ============================================================
# CELL 56 — VALIDATE FINAL CANDIDATES
# ============================================================
# Mục đích:
# - Kiểm tra final_candidates_100.jsonl sau khi combine.
# - Xác nhận JSONL hợp lệ.
# - Kiểm tra đủ 9,667 candidate records.
# - Kiểm tra schema.
# - Kiểm tra candidate thực tế có nội dung.
# - Hiển thị candidate samples.
# - Kiểm tra đủ 40 bug/hunk.
# - Kiểm tra phân bố candidate theo checkpoint.
# - Thống kê exact_match và empty patches.
#
# PHẠM VI STAGE 0:
# - Artifact validation.
# - Candidate validation.
# - KHÔNG chạy pytest.
# - KHÔNG chạy official repair validator.
# - KHÔNG đánh giá plausible/parsable.
# - KHÔNG tính repair rate.
# - KHÔNG thực hiện checkpoint ensemble.
# ============================================================

import json
from collections import Counter
from pathlib import Path


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

OUTPUT = Path(
    "/kaggle/working/MultiMend28/"
    "outputs/QuixBugs-Python/"
    "outputs-multimend/"
    "final_candidates_100.jsonl"
)

EXPECTED_RECORDS = 9667
EXPECTED_RAW_CANDIDATES = 20000
EXPECTED_BUGIDS = 40
EXPECTED_BUG_HUNKS = 40
EXPECTED_SEED = 42

EXPECTED_CONFIG_HASH = (
    "sha256:"
    "c0c516a4f019eabf3b648e39a901e7b81571636ba97dbc31ce8d64063962eaaa"
)

EXPECTED_CHECKPOINTS = {
    "checkpoint-348600",
    "checkpoint-406700",
    "checkpoint-464800",
    "checkpoint-522900",
    "checkpoint-581008",
    "manual",
}


# ------------------------------------------------------------
# Header
# ------------------------------------------------------------

print("=" * 70)
print("STAGE 0 — FINAL CANDIDATES VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# Check file
# ------------------------------------------------------------

assert OUTPUT.exists(), (
    f"Không tìm thấy file: {OUTPUT}"
)

print("\nInput:")
print(OUTPUT)


# ------------------------------------------------------------
# Load JSONL
# ------------------------------------------------------------

records = []

with OUTPUT.open(
    "r",
    encoding="utf-8"
) as file:

    for line_number, line in enumerate(
        file,
        start=1
    ):

        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)

        except json.JSONDecodeError as exc:

            raise AssertionError(
                f"Invalid JSON at line {line_number}: {exc}"
            )

        records.append(record)


# ------------------------------------------------------------
# Record count
# ------------------------------------------------------------

print(
    "\nTotal records:",
    len(records)
)

assert len(records) == EXPECTED_RECORDS, (
    f"Expected {EXPECTED_RECORDS}, "
    f"got {len(records)}"
)

print(
    "Record count validation: PASS"
)


# ------------------------------------------------------------
# Schema validation
# ------------------------------------------------------------

required_keys = {
    "checkpoint",
    "decoded_sequences",
    "sequences_scores",
    "bugid",
    "hunk",
    "source",
    "target",
    "normalized_patch",
    "normalized_source",
    "normalized_target",
    "rank",
    "exact_match",
    "correct",
}

for index, record in enumerate(records):

    missing = (
        required_keys
        - set(record.keys())
    )

    assert not missing, (
        f"Record {index} missing keys: {missing}"
    )

print(
    "Schema validation: PASS"
)


# ============================================================
# CANDIDATE CONTENT VALIDATION
# ============================================================

non_empty_candidates = 0
empty_candidates = 0

for record in records:

    patch = str(
        record["decoded_sequences"]
    ).strip()

    if patch:
        non_empty_candidates += 1
    else:
        empty_candidates += 1


print("\nCandidate content:")
print(
    "  Total candidate records:",
    len(records)
)

print(
    "  Non-empty candidates:",
    non_empty_candidates
)

print(
    "  Empty candidates:",
    empty_candidates
)

assert non_empty_candidates > 0, (
    "No non-empty candidates found."
)

assert (
    non_empty_candidates + empty_candidates
    == len(records)
)

print(
    "Candidate content validation: PASS"
)


# ============================================================
# DISPLAY REAL GENERATED CANDIDATES
# ============================================================

print("\n" + "-" * 70)
print("GENERATED CANDIDATE SAMPLES")
print("-" * 70)

MAX_DISPLAY = 20
shown = 0

for record in records:

    patch = str(
        record["decoded_sequences"]
    ).strip()

    # Chỉ hiển thị candidate có patch thực tế
    if not patch:
        continue

    print(
        f"[{shown + 1}] "
        f"bugid={record['bugid']} | "
        f"hunk={record['hunk']} | "
        f"checkpoint={record['checkpoint']} | "
        f"rank={record['rank']} | "
        f"score={record['sequences_scores']} | "
        f"exact={record['exact_match']}"
    )

    print(
        f"    patch: {patch!r}"
    )

    shown += 1

    if shown >= MAX_DISPLAY:
        break


print(
    f"\nDisplayed {shown} non-empty candidate samples."
)

assert shown > 0, (
    "No candidate sample could be displayed."
)


# ============================================================
# CHECKPOINT DISTRIBUTION
# ============================================================

checkpoint_counts = Counter(
    record["checkpoint"]
    for record in records
)

print("\n" + "-" * 70)
print("CHECKPOINT DISTRIBUTION")
print("-" * 70)

for checkpoint, count in sorted(
    checkpoint_counts.items()
):

    print(
        f"  {checkpoint}: {count}"
    )


# ------------------------------------------------------------
# Check expected checkpoints
# ------------------------------------------------------------

actual_checkpoints = set(
    checkpoint_counts.keys()
)

missing_checkpoints = (
    EXPECTED_CHECKPOINTS
    - actual_checkpoints
)

assert not missing_checkpoints, (
    f"Missing checkpoints: {missing_checkpoints}"
)

print(
    "\nCheckpoint presence: PASS"
)


# ============================================================
# BUG / HUNK COVERAGE
# ============================================================

bugids = {
    record["bugid"]
    for record in records
}

bug_hunks = {
    (
        record["bugid"],
        record["hunk"]
    )
    for record in records
}

print("\n" + "-" * 70)
print("BUG / HUNK COVERAGE")
print("-" * 70)

print(
    "Unique bugids:",
    len(bugids)
)

print(
    "Bug/hunk combinations:",
    len(bug_hunks)
)

assert len(bugids) == EXPECTED_BUGIDS, (
    f"Expected {EXPECTED_BUGIDS} bugids, "
    f"got {len(bugids)}"
)

assert len(bug_hunks) == EXPECTED_BUG_HUNKS, (
    f"Expected {EXPECTED_BUG_HUNKS} bug/hunks, "
    f"got {len(bug_hunks)}"
)

print(
    "Bug coverage: PASS"
)

print(
    "Bug/hunk coverage: PASS"
)


# ============================================================
# EXACT MATCH
# ============================================================

exact_matches = sum(
    bool(record["exact_match"])
    for record in records
)

print(
    "\nExact-match candidates:",
    exact_matches
)


# ============================================================
# EMPTY PATCH
# ============================================================

empty_patches = sum(
    str(record["normalized_patch"]) == ""
    for record in records
)

print(
    "Empty patches:",
    empty_patches
)


# ============================================================
# REPRODUCIBILITY
# ============================================================

print("\n" + "-" * 70)
print("REPRODUCIBILITY METADATA")
print("-" * 70)

print(
    "Seed                :",
    EXPECTED_SEED
)

print(
    "Config hash         :",
    EXPECTED_CONFIG_HASH
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("STAGE 0 ARTIFACT SUMMARY")
print("=" * 70)

print(
    f"Raw candidates        : {EXPECTED_RAW_CANDIDATES:,}"
)

print(
    f"Final candidates      : {len(records):,}"
)

print(
    f"Deduplication         : "
    f"{EXPECTED_RAW_CANDIDATES:,} → {len(records):,}"
)

print(
    f"Non-empty candidates  : {non_empty_candidates:,}"
)

print(
    f"Empty candidates      : {empty_candidates:,}"
)

print(
    f"Unique bugids         : {len(bugids)}"
)

print(
    f"Bug/hunk combinations : {len(bug_hunks)}"
)

print(
    f"Exact-match candidates: {exact_matches}"
)


# ============================================================
# STAGE 0 SCOPE
# ============================================================

print("\n" + "=" * 70)
print("STAGE 0 SCOPE")
print("=" * 70)

print(
    "Artifact validation      : PASS"
)

print(
    "Candidate validation     : PASS"
)

print(
    "Candidate content check : PASS"
)

print(
    "Bug coverage             : PASS"
)

print(
    "Schema validation        : PASS"
)

print(
    "Pytest evaluation        : NOT PERFORMED"
)

print(
    "Full repair evaluation  : NOT PERFORMED"
)

print(
    "Repair rate             : NOT COMPUTED"
)

print(
    "Checkpoint ensemble     : NOT PERFORMED"
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("✅ STAGE 0 BASELINE INFERENCE: PASS")
print("=" * 70)

print(
    "Raw candidates       : 20,000"
)

print(
    "Final candidates     : 9,667"
)

print(
    "Non-empty candidates :",
    non_empty_candidates
)

print(
    "Bugids               : 40"
)

print(
    "Bug/hunks            : 40/40"
)

print(
    "Seed                 : 42"
)

print(
    "Config hash          : "
    "sha256:c0c516a4f019eabf3b648e39a901e7b81571636ba97dbc31ce8d64063962eaaa"
)

print(
    "\nSTATUS: READY FOR STAGE 1"
)

print("=" * 70)


STAGE 0 — FINAL CANDIDATES VALIDATION

Input:
/kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend/final_candidates_100.jsonl

Total records: 9667
Record count validation: PASS
Schema validation: PASS

Candidate content:
  Total candidate records: 9667
  Non-empty candidates: 9631
  Empty candidates: 36
Candidate content validation: PASS

----------------------------------------------------------------------
GENERATED CANDIDATE SAMPLES
----------------------------------------------------------------------
[1] bugid=bitcount | hunk=0 | checkpoint=checkpoint-406700 | rank=0 | score=-0.364941746 | exact=False
    patch: 'n ^= n'
[2] bugid=bitcount | hunk=0 | checkpoint=checkpoint-464800 | rank=0 | score=-0.4491511881 | exact=False
    patch: 'n = n - 1'
[3] bugid=bitcount | hunk=0 | checkpoint=checkpoint-348600 | rank=1 | score=-0.3517662287 | exact=False
    patch: 'n ^= (n - 1)'
[4] bugid=bitcount | hunk=0 | checkpoint=checkpoint-464800 | rank=1 | score=-0.4701306224 | 

In [62]:
# ============================================================
# CELL 57 — QUIXBUGS-PYTHON PYTEST SMOKE TEST
# ============================================================
# Mục tiêu Stage 0:
#   1. Kiểm tra pytest đã được cài đặt.
#   2. Kiểm tra official QuixBugs-Python infrastructure.
#   3. Kiểm tra pytest có thể collect test.
#
# KHÔNG:
#   - đánh giá candidate patch
#   - chạy full pytest
#   - tính repair rate
#   - thực hiện checkpoint ensemble
#
# Lưu ý:
#   test_bitcount.py đã timeout khi execute.
#   Vì vậy smoke test này dùng --collect-only,
#   tức là chỉ collect test và KHÔNG execute test.
# ============================================================

import subprocess
import sys
from pathlib import Path

REPO = Path("/kaggle/working/MultiMend28")
QUIXBUGS_DIR = REPO / "benchmarks" / "QuixBugs"
TEST_DIR = QUIXBUGS_DIR / "python_testcases"

TIMEOUT_SECONDS = 60

print("=" * 70)
print("QUIXBUGS-PYTHON PYTEST SMOKE TEST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Kiểm tra pytest
# ------------------------------------------------------------

pytest_version = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "--version",
    ],
    capture_output=True,
    text=True,
    timeout=TIMEOUT_SECONDS,
)

if pytest_version.returncode != 0:
    print("❌ pytest check FAILED")
    print(pytest_version.stderr)
    raise RuntimeError("pytest is not available.")

print("pytest:", pytest_version.stdout.strip())

# ------------------------------------------------------------
# 2. Kiểm tra official QuixBugs-Python files
# ------------------------------------------------------------

required_files = [
    QUIXBUGS_DIR / "conftest.py",
    QUIXBUGS_DIR / "tester.py",
    TEST_DIR / "load_testdata.py",
]

print("\nOfficial QuixBugs-Python infrastructure:")

for path in required_files:
    if path.is_file():
        print(f"  PASS: {path}")
    else:
        print(f"  FAIL: {path}")
        raise FileNotFoundError(
            f"Missing official QuixBugs-Python file: {path}"
        )

# ------------------------------------------------------------
# 3. Kiểm tra test files
# ------------------------------------------------------------

test_files = sorted(TEST_DIR.glob("test_*.py"))

print(f"\nQuixBugs-Python test files found: {len(test_files)}")

if not test_files:
    raise FileNotFoundError(
        f"No QuixBugs-Python test files found in {TEST_DIR}"
    )

print("Test discovery: PASS")

# ------------------------------------------------------------
# 4. Pytest collection-only
# ------------------------------------------------------------
# Quan trọng:
# --collect-only chỉ đọc/collect test.
# Không thực thi test nên không bị timeout bởi
# test_bitcount.py hoặc các test benchmark khác.

print("\nRunning pytest --collect-only...")

try:
    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            "--collect-only",
            "-q",
            str(TEST_DIR),
        ],
        cwd=REPO,
        capture_output=True,
        text=True,
        timeout=TIMEOUT_SECONDS,
    )

except subprocess.TimeoutExpired:
    print("\n❌ PYTEST COLLECTION: TIMEOUT")
    print(f"Timeout: {TIMEOUT_SECONDS} seconds")
    raise

# ------------------------------------------------------------
# 5. Hiển thị kết quả collection
# ------------------------------------------------------------

if result.stdout.strip():
    print("\n---------- PYTEST COLLECTION ----------")
    print(result.stdout)

if result.stderr.strip():
    print("\n---------- PYTEST STDERR ----------")
    print(result.stderr)

print("\nPytest collection return code:", result.returncode)

# ------------------------------------------------------------
# 6. Kết luận Stage 0 smoke test
# ------------------------------------------------------------

if result.returncode == 0:

    print("\n" + "=" * 70)
    print("✅ QUIXBUGS-PYTHON PYTEST SMOKE TEST: PASS")
    print("=" * 70)

    print("pytest installation       : PASS")
    print("official infrastructure   : PASS")
    print("test discovery            : PASS")
    print("pytest collection         : PASS")

    print("\nIMPORTANT:")
    print("- No repair candidate was evaluated.")
    print("- No full pytest execution was performed.")
    print("- No checkpoint ensemble was performed.")
    print("- Full repair evaluation remains a later stage.")

else:

    print("\n" + "=" * 70)
    print("❌ QUIXBUGS-PYTHON PYTEST SMOKE TEST: FAIL")
    print("=" * 70)

    raise RuntimeError(
        "QuixBugs-Python pytest collection failed."
    )


QUIXBUGS-PYTHON PYTEST SMOKE TEST
pytest: pytest 8.3.2

Official QuixBugs-Python infrastructure:
  PASS: /kaggle/working/MultiMend28/benchmarks/QuixBugs/conftest.py
  PASS: /kaggle/working/MultiMend28/benchmarks/QuixBugs/tester.py
  PASS: /kaggle/working/MultiMend28/benchmarks/QuixBugs/python_testcases/load_testdata.py

QuixBugs-Python test files found: 40
Test discovery: PASS

Running pytest --collect-only...

---------- PYTEST COLLECTION ----------
benchmarks/QuixBugs/python_testcases/test_bitcount.py::test_bitcount[input_data0-7]
benchmarks/QuixBugs/python_testcases/test_bitcount.py::test_bitcount[input_data1-1]
benchmarks/QuixBugs/python_testcases/test_bitcount.py::test_bitcount[input_data2-9]
benchmarks/QuixBugs/python_testcases/test_bitcount.py::test_bitcount[input_data3-3]
benchmarks/QuixBugs/python_testcases/test_bitcount.py::test_bitcount[input_data4-3]
benchmarks/QuixBugs/python_testcases/test_bitcount.py::test_bitcount[input_data5-4]
benchmarks/QuixBugs/python_testcases/test

In [68]:
# ============================================================
# CELL 58 — STAGE 0 FINAL BASELINE REPORT
# ============================================================
# Chốt toàn bộ artifact Baseline Inference.
#
# Stage 0 KHÔNG:
#   - evaluate candidate patches
#   - chạy full pytest
#   - tính repair rate
#   - checkpoint ensemble
#
# Stage 1 sẽ sử dụng:
#   final_candidates_100.jsonl
# ============================================================

from pathlib import Path
import json


REPO = Path(
    "/kaggle/working/MultiMend28"
)

OUTPUT_DIR = (
    REPO
    / "outputs"
    / "QuixBugs-Python"
    / "outputs-multimend"
)

INPUT_FILE = (
    OUTPUT_DIR
    / "generated_input.jsonl"
)

RAW_FILE = (
    OUTPUT_DIR
    / "sequences_100.jsonl"
)

FINAL_FILE = (
    OUTPUT_DIR
    / "final_candidates_100.jsonl"
)

MANIFEST_FILE = (
    OUTPUT_DIR
    / "baseline_manifest.json"
)


print("=" * 70)
print("STAGE 0 — FINAL BASELINE REPORT")
print("=" * 70)


# ------------------------------------------------------------
# Artifact checks
# ------------------------------------------------------------

artifacts = {
    "generated_input.jsonl": INPUT_FILE,
    "sequences_100.jsonl": RAW_FILE,
    "final_candidates_100.jsonl": FINAL_FILE,
    "baseline_manifest.json": MANIFEST_FILE,
}

for name, path in artifacts.items():

    status = (
        "PASS"
        if path.exists()
        else "MISSING"
    )

    print(
        f"{name:<30} {status}"
    )

    assert path.exists(), (
        f"Missing artifact: {path}"
    )


# ------------------------------------------------------------
# Manifest
# ------------------------------------------------------------

with MANIFEST_FILE.open(
    "r",
    encoding="utf-8"
) as f:

    manifest = json.load(f)


print("\nManifest:")
print(
    json.dumps(
        manifest,
        indent=2,
        ensure_ascii=False,
    )
)


# ------------------------------------------------------------
# Final candidate count
# ------------------------------------------------------------

with FINAL_FILE.open(
    "r",
    encoding="utf-8"
) as f:

    final_count = sum(
        1
        for line in f
        if line.strip()
    )

print(
    "\nFinal candidates:",
    final_count
)

assert final_count == 9667


# ------------------------------------------------------------
# Stage 0 conclusion
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("✅ STAGE 0 BASELINE: COMPLETE")
print("=" * 70)

print("""
Dataset              : QuixBugs-Python
Model                : multimend-codet5-small
Seed                 : 42

Checkpoints          : 5
Checkpoint selection : trainer_state.global_step

Raw candidates       : 20,000
Combined candidates  : 9,667
Bugids               : 40
Bug/hunks            : 40

Input JSONL          : PASS
Raw output JSONL     : PASS
Final output JSONL   : PASS
Manifest             : PASS
Pytest smoke         : PASS

Repair evaluation    : NOT PERFORMED
Repair rate          : NOT COMPUTED
Checkpoint ensemble  : NOT PERFORMED

STATUS               : READY FOR STAGE 1
""")

print("=" * 70)


STAGE 0 — FINAL BASELINE REPORT
generated_input.jsonl          PASS
sequences_100.jsonl            PASS
final_candidates_100.jsonl     PASS
baseline_manifest.json         PASS

Manifest:
{
  "experiment": "MultiMend28 baseline inference",
  "config": {
    "dataset": "QuixBugs-Python",
    "model_name": "multimend-codet5-small",
    "seed": 42,
    "max_input_length": 512,
    "max_target_length": 256,
    "min_target_length": 0,
    "beam_size": 100,
    "num_return_sequences": 100,
    "batch_size": 1,
    "num_checkpoints": 5,
    "device": "cuda",
    "input_jsonl": "/kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend/generated_input.jsonl",
    "output_jsonl": "/kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend/sequences_100.jsonl",
    "checkpoints": [
      {
        "name": "checkpoint-348600",
        "global_step": 348600
      },
      {
        "name": "checkpoint-406700",
        "global_step": 406700
      },
      {
        "name": "c

In [60]:
# ================================================================
# # CELL 59 — STAGE 0 — BASELINE INFERENCE FINAL CHECK - CHẠY ALL LẠI ĐỂ KIỂM TRA ( KHÔNG CẦN )
# ================================================================
#
# Purpose:
#   1. Verify baseline inference artifacts.
#   2. Discover checkpoints from trainer_state.json.
#   3. Select the final 5 checkpoints by global_step.
#   4. Verify selected checkpoints against the baseline manifest.
#   5. Check official checkpoint availability if downloadable.
#   6. Validate config hash + seed + input/output JSONL.
#   7. Smoke-test official QuixBugs-Python infrastructure.
#
# Important:
#   - Checkpoint selection is based ONLY on trainer_state.global_step.
#   - Official checkpoint download is informative/non-blocking.
#   - Existing baseline artifacts are NOT regenerated here.
# ================================================================

from pathlib import Path
import json
import hashlib
import subprocess
import sys
import pandas as pd


# ================================================================
# 0. CONFIGURATION
# ================================================================

DATASET = "QuixBugs-Python"
MODEL = "multimend-codet5-small"
SEED = 42

CONFIG_HASH = (
    "sha256:"
    "c0c516a4f019eabf3b648e39a901e7b81571636ba97dbc31ce8d64063962eaaa"
)

BASE_DIR = Path("/kaggle/working/MultiMend28")

OUTPUT_DIR = (
    BASE_DIR
    / "outputs"
    / DATASET
    / "outputs-multimend"
)

MODEL_DIR = (
    BASE_DIR
    / "models"
    / MODEL
)

QUIXBUGS_DIR = (
    BASE_DIR
    / "benchmarks"
    / "QuixBugs"
)

# ------------------------------------------------
# Baseline artifacts
# ------------------------------------------------

INPUT_JSONL = OUTPUT_DIR / "generated_input.jsonl"
RAW_OUTPUT_JSONL = OUTPUT_DIR / "sequences_100.jsonl"
FINAL_OUTPUT_JSONL = OUTPUT_DIR / "final_candidates_100.jsonl"
MANIFEST_JSON = OUTPUT_DIR / "baseline_manifest.json"

# ------------------------------------------------
# Expected number of checkpoints
# ------------------------------------------------

N_CHECKPOINTS = 5

# ------------------------------------------------
# Expected baseline statistics
# ------------------------------------------------

EXPECTED_RAW_COUNT = 20_000
EXPECTED_FINAL_COUNT = 9_667
EXPECTED_BUG_COUNT = 40
EXPECTED_HUNK_COUNT = 40

# ------------------------------------------------
# Official QuixBugs infrastructure
# ------------------------------------------------

OFFICIAL_FILES = [
    QUIXBUGS_DIR / "conftest.py",
    QUIXBUGS_DIR / "tester.py",
    QUIXBUGS_DIR / "python_testcases" / "load_testdata.py",
]

# ================================================================
# 1. HELPERS
# ================================================================

def print_section(title):
    print()
    print("=" * 70)
    print(title)
    print("=" * 70)


def check_file(path, label=None):
    label = label or path.name

    ok = path.exists() and path.stat().st_size > 0

    print(
        f"{label:<40}: "
        f"{'PASS' if ok else 'FAIL'}"
    )

    if not ok:
        print(f"  -> Missing/empty: {path}")

    return ok


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def read_jsonl(path):
    return pd.read_json(
        path,
        orient="records",
        lines=True,
    )


# ================================================================
# 2. INITIAL CONFIGURATION REPORT
# ================================================================

print_section("STAGE 0 — BASELINE CONFIGURATION")

print(f"Dataset              : {DATASET}")
print(f"Model                : {MODEL}")
print(f"Seed                 : {SEED}")
print(f"Config hash          : {CONFIG_HASH}")
print(f"Base directory       : {BASE_DIR}")
print(f"Output directory     : {OUTPUT_DIR}")
print(f"Model directory      : {MODEL_DIR}")
print(f"QuixBugs directory   : {QUIXBUGS_DIR}")

print()
print("Checkpoint policy:")
print("  Selection field    : trainer_state.global_step")
print("  Number selected    :", N_CHECKPOINTS)
print("  Ordering           : ascending global_step")
print("  Final checkpoints  : last 5")


# ================================================================
# 3. DISCOVER CHECKPOINTS FROM trainer_state.json
# ================================================================

print_section("CHECKPOINT DISCOVERY")

discovered_checkpoints = []

if MODEL_DIR.exists():
    checkpoint_dirs = sorted(
        MODEL_DIR.glob("checkpoint-*")
    )
else:
    checkpoint_dirs = []

print(f"Checkpoint directories found: {len(checkpoint_dirs)}")

for ckpt_dir in checkpoint_dirs:

    trainer_state = ckpt_dir / "trainer_state.json"

    if not trainer_state.exists():
        print(
            f"  {ckpt_dir.name:<30}: "
            "SKIP — trainer_state.json missing"
        )
        continue

    try:
        state = load_json(trainer_state)
        global_step = state.get("global_step")

        if global_step is None:
            print(
                f"  {ckpt_dir.name:<30}: "
                "SKIP — global_step missing"
            )
            continue

        discovered_checkpoints.append({
            "name": ckpt_dir.name,
            "path": str(ckpt_dir),
            "global_step": int(global_step),
            "trainer_state": str(trainer_state),
        })

        print(
            f"  {ckpt_dir.name:<30}: "
            f"global_step={int(global_step):,}"
        )

    except Exception as e:
        print(
            f"  {ckpt_dir.name:<30}: "
            f"FAIL — {type(e).__name__}: {e}"
        )


# ================================================================
# 4. SELECT FINAL 5 CHECKPOINTS
# ================================================================

print_section("CHECKPOINT SELECTION")

discovered_checkpoints = sorted(
    discovered_checkpoints,
    key=lambda x: x["global_step"],
)

if len(discovered_checkpoints) < N_CHECKPOINTS:
    selected_checkpoints = []
    checkpoint_selection_ok = False

    print(
        f"FAIL: only {len(discovered_checkpoints)} valid "
        f"checkpoints discovered; need {N_CHECKPOINTS}."
    )

else:
    selected_checkpoints = discovered_checkpoints[-N_CHECKPOINTS:]
    checkpoint_selection_ok = True

    print(
        f"Selected final {N_CHECKPOINTS} checkpoints "
        "using trainer_state.global_step:"
    )

    for rank, ckpt in enumerate(
        selected_checkpoints,
        start=1,
    ):
        print(
            f"  {rank}. "
            f"{ckpt['name']:<30} "
            f"global_step={ckpt['global_step']:,}"
        )


# ================================================================
# 5. EXPECTED CHECKPOINTS
# ================================================================

EXPECTED_CHECKPOINTS = [
    ("checkpoint-348600", 348600),
    ("checkpoint-406700", 406700),
    ("checkpoint-464800", 464800),
    ("checkpoint-522900", 522900),
    ("checkpoint-581008", 581008),
]

actual_checkpoints = [
    (
        x["name"],
        x["global_step"],
    )
    for x in selected_checkpoints
]

checkpoint_match_ok = (
    actual_checkpoints == EXPECTED_CHECKPOINTS
)

print()
print(
    f"{'Checkpoint selection validation':<40}: "
    f"{'PASS' if checkpoint_match_ok else 'FAIL'}"
)

if not checkpoint_match_ok:
    print()
    print("Expected:")
    for item in EXPECTED_CHECKPOINTS:
        print(" ", item)

    print("Actual:")
    for item in actual_checkpoints:
        print(" ", item)


# ================================================================
# 6. OFFICIAL CHECKPOINT AVAILABILITY
# ================================================================
#
# This section does NOT fail Stage 0 if an official checkpoint
# cannot be downloaded. It records availability for provenance.
#
# Replace OFFICIAL_CHECKPOINT_URL with the actual official URL
# when known.
# ================================================================

print_section("OFFICIAL CHECKPOINT CHECK")

OFFICIAL_CHECKPOINT_URL = None

official_checkpoint_status = "NOT_CONFIGURED"

if OFFICIAL_CHECKPOINT_URL:

    print("Official checkpoint URL configured:")
    print(f"  {OFFICIAL_CHECKPOINT_URL}")

    try:
        result = subprocess.run(
            [
                "python",
                "-c",
                (
                    "import urllib.request; "
                    f"urllib.request.urlopen('{OFFICIAL_CHECKPOINT_URL}', timeout=20)"
                ),
            ],
            capture_output=True,
            text=True,
            timeout=30,
        )

        if result.returncode == 0:
            official_checkpoint_status = "AVAILABLE"
        else:
            official_checkpoint_status = "UNAVAILABLE"

    except Exception as e:
        official_checkpoint_status = "UNAVAILABLE"

    print(
        f"{'Official checkpoint availability':<40}: "
        f"{official_checkpoint_status}"
    )

else:
    print(
        f"{'Official checkpoint availability':<40}: "
        "NOT CONFIGURED"
    )

print(
    "NOTE: official checkpoint availability is "
    "informational and does not invalidate Stage 0."
)


# ================================================================
# 7. BASELINE ARTIFACT CHECK
# ================================================================

print_section("BASELINE ARTIFACT VALIDATION")

input_ok = check_file(
    INPUT_JSONL,
    "generated_input.jsonl",
)

raw_ok = check_file(
    RAW_OUTPUT_JSONL,
    "sequences_100.jsonl",
)

final_ok = check_file(
    FINAL_OUTPUT_JSONL,
    "final_candidates_100.jsonl",
)

manifest_ok = check_file(
    MANIFEST_JSON,
    "baseline_manifest.json",
)


# ================================================================
# 8. MANIFEST VALIDATION
# ================================================================

print_section("MANIFEST VALIDATION")

manifest = None
manifest_ok_content = False

if MANIFEST_JSON.exists():

    try:
        manifest = load_json(MANIFEST_JSON)

        cfg = manifest.get("config", {})

        checks = {
            "dataset": cfg.get("dataset") == DATASET,
            "model_name": cfg.get("model_name") == MODEL,
            "seed": cfg.get("seed") == SEED,
            "config_hash": cfg.get("config_hash") == CONFIG_HASH,
            "num_checkpoints": cfg.get("num_checkpoints")
            == N_CHECKPOINTS,
        }

        for key, ok in checks.items():
            print(
                f"{key:<40}: "
                f"{'PASS' if ok else 'FAIL'}"
            )

        manifest_ok_content = all(checks.values())

    except Exception as e:
        print(
            f"{'Manifest JSON parsing':<40}: FAIL"
        )
        print(f"  -> {type(e).__name__}: {e}")

else:
    print(
        f"{'Manifest configuration':<40}: FAIL"
    )


# ================================================================
# 9. MANIFEST CHECKPOINT VALIDATION
# ================================================================

manifest_checkpoint_ok = False

if manifest is not None:

    selected = manifest.get(
        "selected_checkpoints",
        [],
    )

    manifest_actual = [
        (
            x.get("name"),
            x.get("global_step"),
        )
        for x in selected
    ]

    manifest_checkpoint_ok = (
        manifest_actual == EXPECTED_CHECKPOINTS
    )

print(
    f"{'Manifest checkpoint selection':<40}: "
    f"{'PASS' if manifest_checkpoint_ok else 'FAIL'}"
)


# ================================================================
# 10. INPUT JSONL VALIDATION
# ================================================================

print_section("INPUT JSONL VALIDATION")

input_count = None
input_columns = []

if INPUT_JSONL.exists():

    try:
        input_df = read_jsonl(INPUT_JSONL)

        input_count = len(input_df)
        input_columns = list(input_df.columns)

        print(
            f"{'Input records':<40}: "
            f"{input_count:,}"
        )

        print(
            f"{'Input columns':<40}: "
            f"{', '.join(input_columns)}"
        )

        print(
            f"{'Input JSONL parsing':<40}: PASS"
        )

    except Exception as e:
        print(
            f"{'Input JSONL parsing':<40}: FAIL"
        )
        print(f"  -> {type(e).__name__}: {e}")


# ================================================================
# 11. RAW OUTPUT VALIDATION
# ================================================================

print_section("RAW BASELINE OUTPUT VALIDATION")

raw_count = None
checkpoint_counts = {}

if RAW_OUTPUT_JSONL.exists():

    try:

        raw_df = read_jsonl(RAW_OUTPUT_JSONL)

        raw_count = len(raw_df)

        print(
            f"{'Raw candidates':<40}: "
            f"{raw_count:,}"
        )

        if "checkpoint" in raw_df.columns:

            checkpoint_counts = (
                raw_df["checkpoint"]
                .value_counts()
                .to_dict()
            )

            print()
            print("Candidates per checkpoint:")

            for name, step in EXPECTED_CHECKPOINTS:

                count = checkpoint_counts.get(
                    name,
                    0,
                )

                print(
                    f"  {name:<30}: "
                    f"{count:,}"
                )

        raw_count_ok = (
            raw_count == EXPECTED_RAW_COUNT
        )

        print(
            f"{'Raw candidate count':<40}: "
            f"{'PASS' if raw_count_ok else 'FAIL'}"
        )

    except Exception as e:

        raw_count_ok = False

        print(
            f"{'Raw JSONL parsing':<40}: FAIL"
        )

        print(
            f"  -> {type(e).__name__}: {e}"
        )

else:
    raw_count_ok = False


# ================================================================
# 12. FINAL CANDIDATE VALIDATION
# ================================================================

print_section("FINAL CANDIDATE VALIDATION")

final_count = None
bug_count = None
hunk_count = None

if FINAL_OUTPUT_JSONL.exists():

    try:

        final_df = read_jsonl(
            FINAL_OUTPUT_JSONL
        )

        final_count = len(final_df)

        if "bugid" in final_df.columns:
            bug_count = (
                final_df["bugid"]
                .nunique()
            )

        if {
            "bugid",
            "hunk",
        }.issubset(final_df.columns):

            hunk_count = (
                final_df[
                    ["bugid", "hunk"]
                ]
                .drop_duplicates()
                .shape[0]
            )

        print(
            f"{'Final candidates':<40}: "
            f"{final_count:,}"
        )

        print(
            f"{'Unique bugids':<40}: "
            f"{bug_count}"
        )

        print(
            f"{'Bug/hunk combinations':<40}: "
            f"{hunk_count}"
        )

    except Exception as e:

        print(
            f"{'Final JSONL parsing':<40}: FAIL"
        )

        print(
            f"  -> {type(e).__name__}: {e}"
        )


final_count_ok = (
    final_count == EXPECTED_FINAL_COUNT
)

bug_count_ok = (
    bug_count == EXPECTED_BUG_COUNT
)

hunk_count_ok = (
    hunk_count == EXPECTED_HUNK_COUNT
)

print()
print(
    f"{'Final candidate count':<40}: "
    f"{'PASS' if final_count_ok else 'FAIL'}"
)

print(
    f"{'Unique bugid count':<40}: "
    f"{'PASS' if bug_count_ok else 'FAIL'}"
)

print(
    f"{'Bug/hunk count':<40}: "
    f"{'PASS' if hunk_count_ok else 'FAIL'}"
)


# ================================================================
# 13. QUIXBUGS OFFICIAL INFRASTRUCTURE SMOKE
# ================================================================

print_section("QUIXBUGS-PYTHON OFFICIAL SMOKE")

official_files_status = {}

for path in OFFICIAL_FILES:

    ok = path.exists()

    official_files_status[path.name] = ok

    print(
        f"{path.name:<40}: "
        f"{'PASS' if ok else 'FAIL'}"
    )

official_ok = all(
    official_files_status.values()
)

# ------------------------------------------------
# Discover test files
# ------------------------------------------------

test_dir = (
    QUIXBUGS_DIR
    / "python_testcases"
)

test_files = sorted(
    test_dir.glob("test_*.py")
) if test_dir.exists() else []

print()
print(
    f"{'Test files discovered':<40}: "
    f"{len(test_files)}"
)

test_file_count_ok = (
    len(test_files) == 40
)

print(
    f"{'Expected 40 test files':<40}: "
    f"{'PASS' if test_file_count_ok else 'FAIL'}"
)


# ================================================================
# 14. PYTEST COLLECTION
# ================================================================

print_section("PYTEST COLLECTION")

pytest_collection_ok = False
pytest_collected = None
pytest_output = ""

if test_dir.exists():

    try:

        result = subprocess.run(
            [
                sys.executable,
                "-m",
                "pytest",
                str(test_dir),
                "--collect-only",
                "-q",
            ],
            cwd=str(QUIXBUGS_DIR),
            capture_output=True,
            text=True,
            timeout=300,
        )

        stdout = result.stdout or ""
        stderr = result.stderr or ""

        pytest_output = stdout + "\n" + stderr

        # --------------------------------------------------------
        # IMPORTANT:
        # Pytest may report either:
        #
        #   278 tests collected in 0.08s
        #
        # or:
        #
        #   278 items collected
        #
        # Support both formats.
        # --------------------------------------------------------

        import re

        collection_patterns = [
            r"(\d+)\s+tests?\s+collected",
            r"(\d+)\s+items?\s+collected",
            r"collected\s+(\d+)\s+tests?",
            r"collected\s+(\d+)\s+items?",
        ]

        for pattern in collection_patterns:

            match = re.search(
                pattern,
                pytest_output,
                flags=re.IGNORECASE,
            )

            if match:
                pytest_collected = int(
                    match.group(1)
                )
                break

        # --------------------------------------------------------
        # Do NOT print every collected test unless debugging.
        # This keeps the Stage 0 report compact.
        # --------------------------------------------------------

        print(
            f"Pytest return code                 : "
            f"{result.returncode}"
        )

        print(
            f"Tests collected                    : "
            f"{pytest_collected if pytest_collected is not None else 'N/A'}"
        )

        pytest_collection_ok = (
            result.returncode == 0
            and pytest_collected == 278
        )

        print(
            f"{'Pytest collection':<40}: "
            f"{'PASS' if pytest_collection_ok else 'FAIL'}"
        )

        # --------------------------------------------------------
        # If parsing failed, show pytest's final summary
        # instead of dumping hundreds of test names.
        # --------------------------------------------------------

        if pytest_collected is None:

            print()
            print("Pytest output tail:")
            print(
                "\n".join(
                    pytest_output.strip().splitlines()[-10:]
                )
            )

    except subprocess.TimeoutExpired:

        pytest_collection_ok = False

        print(
            f"{'Pytest collection':<40}: FAIL"
        )

        print(
            "Reason: pytest collection timed out."
        )

    except Exception as e:

        pytest_collection_ok = False

        print(
            f"{'Pytest collection':<40}: FAIL"
        )

        print(
            f"Reason: {type(e).__name__}: {e}"
        )

else:

    print(
        f"{'Pytest collection':<40}: FAIL"
    )

    print(
        "Reason: QuixBugs python_testcases directory missing."
    )

# ================================================================
# 15. BASELINE HASH / PROVENANCE REPORT
# ================================================================

print_section("BASELINE PROVENANCE")

print(f"Dataset              : {DATASET}")
print(f"Model                : {MODEL}")
print(f"Seed                 : {SEED}")
print(f"Config hash          : {CONFIG_HASH}")
print(
    "Checkpoint selection : "
    "trainer_state.global_step"
)
print(f"Checkpoints           : {N_CHECKPOINTS}")

print()
print("Artifacts:")

for path in [
    INPUT_JSONL,
    RAW_OUTPUT_JSONL,
    FINAL_OUTPUT_JSONL,
    MANIFEST_JSON,
]:

    if path.exists():

        print(
            f"  {path.name:<32} "
            f"size={path.stat().st_size:,} bytes"
        )

    else:

        print(
            f"  {path.name:<32} "
            "MISSING"
        )

# ================================================================
# 16. PYTEST COLLECTION
# ================================================================

print_section("PYTEST COLLECTION")

pytest_collection_ok = False
pytest_collected = None
pytest_exit_code = None
pytest_collection_error = None

EXPECTED_TEST_FILES = 40
EXPECTED_TESTS_COLLECTED = 278

try:
    import pytest

    # ------------------------------------------------------------
    # Custom plugin to capture collection statistics
    # ------------------------------------------------------------
    class CollectionStatsPlugin:

        def __init__(self):
            self.tests_collected = 0
            self.collection_errors = 0
            self.session_started = False
            self.session_finished = False

        def pytest_sessionstart(self, session):
            self.session_started = True

        def pytest_collection_finish(self, session):
            self.tests_collected = len(session.items)
            self.collection_errors = len(session.testsfailed)

        def pytest_sessionfinish(self, session, exitstatus):
            self.session_finished = True

    collection_plugin = CollectionStatsPlugin()

    # ------------------------------------------------------------
    # Run collection only
    #
    # --collect-only:
    #   Collect tests without executing them.
    #
    # -q:
    #   Keep output compact.
    # ------------------------------------------------------------
    pytest_args = [
        str(QUIXBUGS_DIR / "python_testcases"),
        "--collect-only",
        "-q",
    ]

    pytest_exit_code = pytest.main(
        pytest_args,
        plugins=[collection_plugin],
    )

    pytest_collected = collection_plugin.tests_collected

    # ------------------------------------------------------------
    # Collection is considered successful when:
    #
    # 1. pytest actually started
    # 2. pytest finished collection
    # 3. no collection error occurred
    # 4. expected number of tests was collected
    #
    # NOTE:
    # We intentionally do NOT require a full pytest run here.
    # Stage 0 only validates the official test infrastructure
    # and collection.
    # ------------------------------------------------------------
    pytest_collection_ok = (
        collection_plugin.session_started
        and collection_plugin.session_finished
        and collection_plugin.collection_errors == 0
        and pytest_collected == EXPECTED_TESTS_COLLECTED
    )

except Exception as exc:

    pytest_collection_error = repr(exc)
    pytest_collection_ok = False

# ------------------------------------------------------------
# Collection report
# ------------------------------------------------------------
print(
    f"{'Expected tests':<40}: "
    f"{EXPECTED_TESTS_COLLECTED}"
)

print(
    f"{'Tests collected':<40}: "
    f"{pytest_collected if pytest_collected is not None else 'N/A'}"
)

print(
    f"{'Pytest exit code':<40}: "
    f"{pytest_exit_code if pytest_exit_code is not None else 'N/A'}"
)

print(
    f"{'Collection errors':<40}: "
    f"{collection_plugin.collection_errors}"
    if 'collection_plugin' in locals()
    else "Collection errors".ljust(40) + ": N/A"
)

print(
    f"{'Pytest collection':<40}: "
    f"{'PASS' if pytest_collection_ok else 'FAIL'}"
)

if pytest_collection_error is not None:
    print(
        f"{'Collection error':<40}: "
        f"{pytest_collection_error}"
    )


# ================================================================
# 17. FINAL STAGE 0 CHECKS
# ================================================================

# ------------------------------------------------------------
# Keep official checkpoint informational.
#
# "NOT_CONFIGURED" must NOT invalidate Stage 0 because the
# experiment can legitimately proceed with the locally available
# checkpoints.
# ------------------------------------------------------------
official_checkpoint_ok = True


# ------------------------------------------------------------
# Build final check dictionary
# ------------------------------------------------------------
stage0_checks = {
    "generated_input.jsonl": input_ok,
    "sequences_100.jsonl": raw_ok,
    "final_candidates_100.jsonl": final_ok,
    "baseline_manifest.json": manifest_ok,

    "manifest configuration": manifest_ok_content,

    "checkpoint discovery": (
        len(discovered_checkpoints) >= 5
        if "discovered_checkpoints" in locals()
        else len(selected_checkpoints) >= 5
    ),

    "checkpoint selection": checkpoint_ok,

    "manifest checkpoints": checkpoint_ok,

    "raw candidate count": (
        raw_count == 20_000
    ),

    "final candidate count": (
        final_count == 9_667
    ),

    "unique bugid count": (
        bug_count == 40
    ),

    "bug/hunk count": (
        hunk_count == 40
    ),

    "official QuixBugs infrastructure": (
        official_ok
    ),

    "40 test files": (
        len(test_files) == EXPECTED_TEST_FILES
    ),

    "pytest collection": (
        pytest_collection_ok
    ),
}


# ------------------------------------------------------------
# Final Stage 0 decision
#
# Official checkpoint availability is intentionally excluded
# from the blocking checks.
# ------------------------------------------------------------
stage0_pass = all(stage0_checks.values())


# ================================================================
# 18. FINAL SUMMARY TABLE
# ================================================================

print_section("STAGE 0 — FINAL SUMMARY")

for name, ok in stage0_checks.items():

    print(
        f"{name:<40}: "
        f"{'PASS' if ok else 'FAIL'}"
    )

print()
print("-" * 70)

if stage0_pass:
    print("✅ STAGE 0 BASELINE: COMPLETE")
else:
    print("❌ STAGE 0 BASELINE: INCOMPLETE")

print("-" * 70)

print()
print("Dataset              :", DATASET)
print("Model                :", MODEL)
print("Seed                 :", SEED)
print("Config hash          :", CONFIG_HASH)

print(
    "Checkpoint selection : "
    "trainer_state.global_step"
)

print(
    "Checkpoints          : "
    f"{len(selected_checkpoints)}"
)

print(
    "Raw candidates       : "
    f"{raw_count:,}"
    if raw_count is not None
    else "Raw candidates       : N/A"
)

print(
    "Final candidates     : "
    f"{final_count:,}"
    if final_count is not None
    else "Final candidates     : N/A"
)

print(
    "Bugids               : "
    f"{bug_count}"
    if bug_count is not None
    else "Bugids               : N/A"
)

print(
    "Bug/hunks            : "
    f"{hunk_count}"
    if hunk_count is not None
    else "Bug/hunks            : N/A"
)

print(
    "Pytest smoke         : "
    f"{'PASS' if pytest_collection_ok else 'FAIL'}"
)

print(
    "Tests collected      : "
    f"{pytest_collected}"
    if pytest_collected is not None
    else "Tests collected      : N/A"
)

print(
    "Official checkpoint  : "
    f"{official_checkpoint_status}"
)

print()
print("Repair candidate evaluation : NOT PERFORMED")
print("Full pytest execution       : NOT PERFORMED")
print("Checkpoint ensemble         : NOT PERFORMED")

print()

if stage0_pass:
    print("STATUS                : READY FOR STAGE 1")
else:
    print("STATUS                : BLOCKED — FIX FAILED CHECKS")

print("=" * 70)


# ================================================================
# 19. MACHINE-READABLE RESULT
# ================================================================

stage0_result = {
    "stage": 0,

    "status": (
        "READY_FOR_STAGE_1"
        if stage0_pass
        else "BLOCKED"
    ),

    "dataset": DATASET,
    "model": MODEL,
    "seed": SEED,
    "config_hash": CONFIG_HASH,

    "checkpoint_selection": {
        "field": "trainer_state.global_step",
        "num_checkpoints": len(selected_checkpoints),
        "ordering": "ascending_global_step",
        "policy": "last_5",
    },

    "selected_checkpoints": [
        {
            "name": x["name"],
            "global_step": x["global_step"],
        }
        for x in selected_checkpoints
    ],

    "artifacts": {
        "generated_input": {
            "path": str(INPUT_JSONL),
            "exists": INPUT_JSONL.exists(),
            "size_bytes": (
                INPUT_JSONL.stat().st_size
                if INPUT_JSONL.exists()
                else None
            ),
            "valid": input_ok,
        },

        "raw_output": {
            "path": str(RAW_OUTPUT_JSONL),
            "exists": RAW_OUTPUT_JSONL.exists(),
            "size_bytes": (
                RAW_OUTPUT_JSONL.stat().st_size
                if RAW_OUTPUT_JSONL.exists()
                else None
            ),
            "valid": raw_ok,
        },

        "final_candidates": {
            "path": str(FINAL_OUTPUT_JSONL),
            "exists": FINAL_OUTPUT_JSONL.exists(),
            "size_bytes": (
                FINAL_OUTPUT_JSONL.stat().st_size
                if FINAL_OUTPUT_JSONL.exists()
                else None
            ),
            "valid": final_ok,
        },

        "baseline_manifest": {
            "path": str(MANIFEST_JSON),
            "exists": MANIFEST_JSON.exists(),
            "size_bytes": (
                MANIFEST_JSON.stat().st_size
                if MANIFEST_JSON.exists()
                else None
            ),
            "valid": manifest_ok,
        },
    },

    "input": {
        "records": (
            len(input_df)
            if "input_df" in locals() and input_df is not None
            else None
        ),
        "columns": (
            list(input_df.columns)
            if "input_df" in locals() and input_df is not None
            else None
        ),
        "jsonl_valid": input_ok,
    },

    "raw_baseline": {
        "candidates": raw_count,
        "expected_candidates": 20_000,

        "candidates_per_checkpoint": {
            name: checkpoint_counts.get(name, 0)
            for name, _ in EXPECTED_CHECKPOINTS
        },

        "count_valid": raw_count == 20_000,
    },

    "final_candidates": {
        "candidates": final_count,
        "expected_candidates": 9_667,

        "unique_bugids": bug_count,
        "expected_bugids": 40,

        "bug_hunk_combinations": hunk_count,
        "expected_bug_hunk_combinations": 40,
    },

    "quixbugs": {
        "official_infrastructure": official_ok,
        "test_files": len(test_files),
        "expected_test_files": EXPECTED_TEST_FILES,

        "pytest_collection": {
            "status": (
                "PASS"
                if pytest_collection_ok
                else "FAIL"
            ),
            "tests_collected": pytest_collected,
            "expected_tests": EXPECTED_TESTS_COLLECTED,
            "exit_code": pytest_exit_code,
            "collection_errors": (
                collection_plugin.collection_errors
                if "collection_plugin" in locals()
                else None
            ),
        },
    },

    "official_checkpoint": {
        "status": official_checkpoint_status,
        "informational_only": True,
    },

    "stage1": {
        "repair_candidate_evaluation": "NOT_PERFORMED",
        "full_pytest": "NOT_PERFORMED",
        "checkpoint_ensemble": "NOT_PERFORMED",
    },

    "checks": stage0_checks,
}


# ================================================================
# 20. SAVE MACHINE-READABLE RESULT
# ================================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

STAGE0_RESULT_JSON = OUTPUT_DIR / "stage0_final_check.json"

with open(
    STAGE0_RESULT_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        stage0_result,
        f,
        indent=2,
        ensure_ascii=False,
    )

print()
print(
    "Machine-readable result saved to:"
)

print(
    f"  {STAGE0_RESULT_JSON}"
)



STAGE 0 — BASELINE CONFIGURATION
Dataset              : QuixBugs-Python
Model                : multimend-codet5-small
Seed                 : 42
Config hash          : sha256:c0c516a4f019eabf3b648e39a901e7b81571636ba97dbc31ce8d64063962eaaa
Base directory       : /kaggle/working/MultiMend28
Output directory     : /kaggle/working/MultiMend28/outputs/QuixBugs-Python/outputs-multimend
Model directory      : /kaggle/working/MultiMend28/models/multimend-codet5-small
QuixBugs directory   : /kaggle/working/MultiMend28/benchmarks/QuixBugs

Checkpoint policy:
  Selection field    : trainer_state.global_step
  Number selected    : 5
  Ordering           : ascending global_step
  Final checkpoints  : last 5

CHECKPOINT DISCOVERY
Checkpoint directories found: 5
  checkpoint-348600             : global_step=348,600
  checkpoint-406700             : global_step=406,700
  checkpoint-464800             : global_step=464,800
  checkpoint-522900             : global_step=522,900
  checkpoint-581008      

**Bư****ớc khởi đầu Train model của Papper******

In [ ]:
# Cell 1 — Kaggle environment
import os, sys, subprocess, textwrap, json, pathlib, shutil, time
from pathlib import Path

print("Python:", sys.version)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch import error:", repr(e))


In [ ]:
# Cell 2 — Clone ONLY the MultiMend repo first.
# We intentionally do not recurse through all benchmark submodules yet.

REPO = Path("/kaggle/working/MultiMend")
if REPO.exists():
    print("Repo already exists:", REPO)
else:
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/h4iku/MultiMend.git",
        str(REPO)
    ], check=True)

os.chdir(REPO)
print("HEAD:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("Repo:", REPO)
print("Top-level:", [p.name for p in REPO.iterdir()])


In [ ]:
# Cell 3 — Inspect the official repo BEFORE running anything
from pathlib import Path

for p in sorted((REPO / "src").rglob("*.py")):
    print(p.relative_to(REPO))

print("\nrequirements.txt:")
print((REPO / "requirements.txt").read_text())


## Important implementation findings

The official generation script uses:

- `max_input_length = 512`
- `max_target_length = 256`
- `beam_size = 100`
- `num_return_sequences = 100`
- `batch_size = 1`
- `num_checkpoints = 5`

It loads the last five `checkpoint-*` directories under `models/multimend-codet5-small`, generates from every checkpoint, and writes `sequences_100.jsonl`.

The context augmentation is local RAG: `all-MiniLM-L6-v2` + persistent ChromaDB, not a paid LLM API.


In [ ]:
# Cell 4 — Install official MultiMend dependencies.
# Do NOT upgrade unrelated Kaggle packages.

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "-r",
        "requirements.txt",
        "more-itertools",
        "pygments",
    ],
    text=True,
)

if result.returncode != 0:
    raise RuntimeError("MultiMend dependency installation failed.")

print("MultiMend dependency installation complete.")

In [ ]:
# Cell 4b — Force the versions required by MultiMend
import subprocess
import sys

packages = [
    "numpy==1.26.4",
    "pandas==2.2.3",
    "torch==2.4.0",
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        *packages,
    ],
    check=True,
)

print("Pinned NumPy/Pandas/Torch versions installed.")

In [ ]:
# Cell 4c — Match torchvision with MultiMend's torch 2.4.0

import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        "torchvision==0.19.0",
    ],
    check=True,
)

print("torchvision 0.19.0 installed.")

In [ ]:
# Fix NumPy for ChromaDB 0.5.5 / MultiMend
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m", "pip", "install",
        "-q",
        "--no-cache-dir",
        "--force-reinstall",
        "numpy==1.26.4",
    ],
    check=True,
)

print("NumPy 1.26.4 installed.")


In [ ]:
import numpy as np

print("NumPy:", np.__version__)


In [ ]:
import torch
import transformers
import datasets
import chromadb
import sentence_transformers
import tree_sitter
import unidiff
import more_itertools
import pygments

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("chromadb:", chromadb.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("tree-sitter:", tree_sitter.__version__ if hasattr(tree_sitter, "__version__") else "unknown")
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Cell 5 — Verify versions and direct imports
import torch, transformers, datasets, chromadb, sentence_transformers
import tree_sitter, unidiff, more_itertools, pygments

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("chromadb:", chromadb.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("tree-sitter:", tree_sitter.__version__ if hasattr(tree_sitter, "__version__") else "unknown")
print("CUDA:", torch.cuda.is_available())


In [24]:
from pathlib import Path

REPO = Path("/kaggle/working/MultiMend")

print("Repo exists:", REPO.exists())
print("Repo:", REPO)

if REPO.exists():
    print("Contents:")
    for p in list(REPO.iterdir())[:20]:
        print(" -", p.name)


Repo exists: False
Repo: /kaggle/working/MultiMend


In [ ]:
# Cell 6 — Download QuixBugs

from pathlib import Path
import subprocess
import shutil

REPO = Path("/kaggle/working/MultiMend")
QB = REPO / "benchmarks" / "QuixBugs"

if not REPO.exists():
    raise FileNotFoundError(
        f"MultiMend repo not found at {REPO}. "
        "Please run the clone cell first."
    )

if not (QB / "python_programs").exists():
    if QB.exists():
        shutil.rmtree(QB)

    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/jkoppel/QuixBugs.git",
        str(QB)
    ], check=True)

print("QuixBugs files:")
for d in ["python_programs", "correct_python_programs", "python_testcases"]:
    p = QB / d
    print(
        d,
        "=>", p.exists(),
        "count=", len(list(p.glob("*.py"))) if p.exists() else 0
    )


In [ ]:
# Check what MultiMend itself actually references
!grep -RniE "from_pretrained|snapshot_download|huggingface|checkpoint" \
    /kaggle/working/MultiMend/src \
    --include="*.py" | head -200


xác định chính xác workflow/config của tác giả

In [ ]:
# Cell 7a — Inspect MultiMend model/config paths

print("=== configs.py ===")
print((REPO / "src" / "configs.py").read_text())

print("\n=== generate_primary_candidates.py (relevant section) ===")
lines = (REPO / "src" / "generate_primary_candidates.py").read_text().splitlines()
for i, line in enumerate(lines):
    if 180 <= i <= 235:
        print(f"{i+1}: {line}")


In [ ]:
# Cell 7b — Inspect existing model/results artifacts

from pathlib import Path

print("=== models/ ===")
models_dir = REPO / "models"

if models_dir.exists():
    for p in models_dir.rglob("*"):
        if p.is_file():
            print(p.relative_to(REPO))
else:
    print("No models directory yet.")

print("\n=== results/ ===")
results_dir = REPO / "results"

for p in results_dir.rglob("*"):
    if p.is_file():
        print(p.relative_to(REPO))


In [ ]:
# # Cell 7 — Download the OFFICIAL MultiMend fine-tuned checkpoint repository.
# # No API key is required for a public Hugging Face repository.

# from huggingface_hub import snapshot_download

# MODEL_REPO = "h4iku/multimend-codet5-small"
# MODEL_DIR = REPO / "models" / "multimend-codet5-small"
# MODEL_DIR.mkdir(parents=True, exist_ok=True)

# snapshot_download(
#     repo_id=MODEL_REPO,
#     local_dir=str(MODEL_DIR),
#     local_dir_use_symlinks=False,
# )

# print("Model snapshot:", MODEL_DIR)
# print("Checkpoint directories:")
# for p in sorted(MODEL_DIR.glob("checkpoint-*")):
#     print(" ", p.name)


In [ ]:
# Cell 8 — Inspect the official MultiMend training configuration

train_file = REPO / "src" / "train_model.py"
lines = train_file.read_text().splitlines()

for i, line in enumerate(lines, start=1):
    print(f"{i:3}: {line}")


tính chính xác số sample + số training steps + save_steps từ chính 4 dataset mà code dùng. (Chạy CPU thôi, không tốn GPU)

In [ ]:
# Cell 9 — Inspect official CoCoNuT training dataset sizes

from datasets import load_dataset

dataset_names = {
    "Python": "h4iku/coconut_python2010_preprocessed",
    "Java": "h4iku/coconut_java2006_preprocessed",
    "JavaScript": "h4iku/coconut_javascript2010_preprocessed",
    "C": "h4iku/coconut_c2005_preprocessed",
}

for prefix, name in dataset_names.items():
    print(f"\nLoading {prefix}: {name}")
    ds = load_dataset(name, split="train")
    print("raw samples:", len(ds))
    print("columns:", ds.column_names)


đã biết raw dataset tổng cộng khoảng 2.64 triệu samples, nhưng code còn filter rất nhiều sample. Cần tính số sample sau filter, vì đó mới quyết định 30 GPU-hours có đủ không.

In [ ]:
# Cell 10 — Reproduce the official filtering step and calculate training steps

from math import ceil
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer

model_checkpoint = "Salesforce/codet5-small"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

dataset_names = {
    "Python": "h4iku/coconut_python2010_preprocessed",
    "Java": "h4iku/coconut_java2006_preprocessed",
    "JavaScript": "h4iku/coconut_javascript2010_preprocessed",
    "C": "h4iku/coconut_c2005_preprocessed",
}

max_input_length = 512
max_target_length = 256


def preprocess_function(prefix, examples):
    inputs = [
        f"{prefix} {ex.strip()} :".replace(
            tokenizer.eos_token, tokenizer.unk_token
        )
        for ex in examples["rem"]
    ]

    targets = [
        ex.strip().replace(
            tokenizer.eos_token, tokenizer.unk_token
        )
        for ex in examples["add"]
    ]

    contexts = [
        " ".join(ex.split()).replace(
            tokenizer.eos_token, tokenizer.unk_token
        )
        for ex in examples["context"]
    ]

    inputs_contexts = [
        f"{src} {ctx}" for src, ctx in zip(inputs, contexts)
    ]

    model_inputs = tokenizer(
        inputs_contexts,
        max_length=max_input_length,
        truncation=True,
    )

    model_inputs["inputs_only"] = tokenizer(inputs).input_ids

    labels = tokenizer(targets)
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


def allowed_length(example):
    return (
        len(example["inputs_only"]) <= max_input_length
        and 2 < len(example["labels"]) <= max_target_length
    )


filtered_datasets = []

for prefix, dataset_name in dataset_names.items():
    print(f"\nProcessing {prefix}...")

    raw_dataset = load_dataset(dataset_name, split="train")

    tokenized = raw_dataset.map(
        lambda ex: preprocess_function(prefix, ex),
        batched=True,
        remove_columns=raw_dataset.column_names,
        desc=f"Tokenizing {prefix}",
    )

    filtered = tokenized.filter(
        allowed_length,
        desc=f"Filtering {prefix}",
    ).remove_columns(["inputs_only"])

    print("Raw:", len(raw_dataset))
    print("After filter:", len(filtered))

    filtered_datasets.append(filtered)


concatenated_dataset = concatenate_datasets(filtered_datasets)

print("\n==============================")
print("FINAL DATASET")
print("==============================")
print("Total training samples:", len(concatenated_dataset))

train_batch_size = 8
epochs = 2

epoch_steps = ceil(len(concatenated_dataset) / train_batch_size)
train_steps = epoch_steps * epochs
save_steps = epoch_steps // 5

print("Steps / epoch:", epoch_steps)
print("Total training steps:", train_steps)
print("Checkpoint save_steps:", save_steps)
print("Expected checkpoints:", 10)


In [ ]:
# Cell 11 — Benchmark official MultiMend training speed
# Uses the exact official model/training batch size.
# Only 200 steps, then estimate full training time.

import time
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

model_checkpoint = "Salesforce/codet5-small"

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

benchmark_args = Seq2SeqTrainingArguments(
    output_dir=str(REPO / "benchmark_train"),
    per_device_train_batch_size=8,
    max_steps=200,
    fp16=True,
    learning_rate=1e-4,
    lr_scheduler_type="constant",
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=benchmark_args,
    train_dataset=concatenated_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting 200-step benchmark...")
start = time.time()

trainer.train()

elapsed = time.time() - start

print("\n==============================")
print("BENCHMARK")
print("==============================")
print(f"200 steps: {elapsed / 60:.2f} minutes")
print(f"Average: {elapsed / 200:.3f} sec/step")

estimated_hours = (elapsed / 200) * 581008 / 3600

print(f"Estimated full 581,008 steps: {estimated_hours:.2f} GPU hours")
print(f"30-hour budget remaining margin: {30 - estimated_hours:.2f} hours")


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/thothanh28/"
    "checkpoint58100/checkpoint-58100"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-58100"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)

In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/honglcl/"
    "checkpoint-116200/checkpoint-116200"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-116200"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/honglcl/"
    "checkpoint174300/checkpoint-174300"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-174300"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/minhhanh123/"
    "checkpoint-232400/checkpoint-232400"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-232400"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)

In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/tramtvt/"
    "checkpoint348600/checkpoint-348600"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-348600"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/tramtvt/checkpoint406700/"
    "checkpoint-406700"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-406700"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/nguynhongbchtrm/"
    "checkpoint464800/checkpoint-464800"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-464800"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
from pathlib import Path
import shutil

src = Path(
    "/kaggle/input/datasets/nguynhongbchtrm/"
    "checkpoint522900/checkpoint-522900"
)

dst = Path(
    "/kaggle/working/MultiMend/models/"
    "multimend-codet5-small/checkpoint-522900"
)

if not src.exists():
    raise FileNotFoundError(f"Checkpoint không tồn tại: {src}")

dst.parent.mkdir(parents=True, exist_ok=True)

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Restored checkpoint:")
print(dst)
print()
print("Files:")

for p in sorted(dst.iterdir()):
    print(" -", p.name)


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for d in dirs:
        if "522900" in d:
            print("FOUND:", os.path.join(root, d))


In [ ]:
import os

print("WORLD_SIZE:", os.environ.get("WORLD_SIZE"))
print("LOCAL_RANK:", os.environ.get("LOCAL_RANK"))
print("RANK:", os.environ.get("RANK"))


mục tiêu giữ nguyên training config của MultiMend và chỉ thêm cơ chế auto-resume, dùng cell này. Nó sẽ tìm checkpoint-* mới nhất trong models/multimend-codet5-small/; nếu chưa có thì train từ đầu, nếu có thì resume từ checkpoint đó.

In [ ]:
# Cell 12 — Official MultiMend training with automatic checkpoint resume
#
# IMPORTANT:
# - Keeps the official MultiMend training configuration unchanged.
# - Only adds automatic checkpoint detection/resume.
# - Run this cell again in a new Kaggle session after restoring the
#   checkpoint directory.
#
# Expected checkpoint layout:
# /kaggle/working/MultiMend/models/multimend-codet5-small/
# ├── checkpoint-58100/
# ├── checkpoint-116200/
# ├── ...
#
# Do NOT delete optimizer.pt, scheduler.pt, trainer_state.json, etc.
# from the checkpoint if you want a proper training resume.

from pathlib import Path
import re
import torch

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    GenerationConfig,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

# ------------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------------

REPO = Path("/kaggle/working/MultiMend")
MODELS_ROOT = REPO / "models"
OUTPUT_DIR = MODELS_ROOT / "multimend-codet5-small"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO)
print("Training output:", OUTPUT_DIR)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)


# ------------------------------------------------------------------
# 2. Auto-detect latest valid checkpoint
# ------------------------------------------------------------------

def get_latest_checkpoint(output_dir: Path):
    checkpoints = []

    for p in output_dir.glob("checkpoint-*"):
        if not p.is_dir():
            continue

        match = re.fullmatch(r"checkpoint-(\d+)", p.name)
        if match:
            step = int(match.group(1))
            checkpoints.append((step, p))

    if not checkpoints:
        return None

    checkpoints.sort(key=lambda x: x[0])

    return checkpoints[-1][1]


latest_checkpoint = get_latest_checkpoint(OUTPUT_DIR)

if latest_checkpoint is None:
    print("\nNo existing checkpoint found.")
    print("Training will start from Salesforce/codet5-small.")
else:
    print("\nLatest checkpoint found:")
    print(" ", latest_checkpoint)

    # Useful sanity check
    required_files = [
        "trainer_state.json",
        "optimizer.pt",
        "scheduler.pt",
    ]

    missing = [
        f for f in required_files
        if not (latest_checkpoint / f).exists()
    ]

    if missing:
        raise RuntimeError(
            "Checkpoint appears incomplete.\n"
            f"Checkpoint: {latest_checkpoint}\n"
            f"Missing files: {missing}\n\n"
            "Do not resume from an incomplete checkpoint."
        )

    print("Checkpoint training state looks complete.")


# ------------------------------------------------------------------
# 3. Official base model
# ------------------------------------------------------------------

MODEL_CHECKPOINT = "Salesforce/codet5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT)


# ------------------------------------------------------------------
# 4. IMPORTANT:
#    Reuse the tokenized/filtered training dataset from Cell 8/earlier.
#
#    This variable should already exist in the notebook:
#
#        concatenated_dataset
#
#    We deliberately do NOT recreate the 2.3M-example preprocessing here.
# ------------------------------------------------------------------

if "concatenated_dataset" not in globals():
    raise RuntimeError(
        "concatenated_dataset is not available.\n"
        "Run the official dataset preprocessing cell first."
    )

print("\nTraining dataset:")
print(concatenated_dataset)


# ------------------------------------------------------------------
# 5. Official MultiMend generation configuration
# ------------------------------------------------------------------

max_input_length = 512
max_target_length = 256

generation_config = GenerationConfig(
    max_length=max_target_length,
    min_length=0,
    early_stopping=True,
    num_beams=5,
)


# ------------------------------------------------------------------
# 6. Official MultiMend training configuration
# ------------------------------------------------------------------

train_batch_size = 8
eval_batch_size = 16

epochs = 2
lr = 1e-4

checkpoints_each_epoch = 5

epoch_steps = (
    len(concatenated_dataset) // train_batch_size
    + int(len(concatenated_dataset) % train_batch_size != 0)
)

train_steps = epoch_steps * epochs

save_steps = epoch_steps // checkpoints_each_epoch

print("\n==============================")
print("OFFICIAL TRAINING CONFIG")
print("==============================")
print("Dataset size:", len(concatenated_dataset))
print("Train batch size:", train_batch_size)
print("Epoch steps:", epoch_steps)
print("Total training steps:", train_steps)
print("Save steps:", save_steps)
print("Epochs:", epochs)
print("Learning rate:", lr)
print("FP16:", True)


# ------------------------------------------------------------------
# 7. Trainer
# ------------------------------------------------------------------

args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),

    learning_rate=lr,

    per_device_train_batch_size=train_batch_size,
    per_device_eval_batch_size=eval_batch_size,

    save_total_limit=epochs * checkpoints_each_epoch,

    max_steps=train_steps,

    save_steps=save_steps,

    predict_with_generate=True,

    fp16=True,

    lr_scheduler_type="constant",

    generation_config=generation_config,

    report_to="tensorboard",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=concatenated_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)


# ------------------------------------------------------------------
# 8. AUTO RESUME
# ------------------------------------------------------------------

print("\n==============================")
print("TRAINING")
print("==============================")

if latest_checkpoint is None:

    print("Starting training from scratch...")
    
    trainer.train(
        resume_from_checkpoint=False
    )

else:

    print(f"Resuming from: {latest_checkpoint}")

    trainer.train(
        resume_from_checkpoint=str(latest_checkpoint)
    )


# ------------------------------------------------------------------
# 9. Final status
# ------------------------------------------------------------------

print("\n==============================")
print("TRAINING SESSION FINISHED")
print("==============================")

latest_checkpoint = get_latest_checkpoint(OUTPUT_DIR)

if latest_checkpoint:
    print("Latest checkpoint:")
    print(latest_checkpoint)

    state_file = latest_checkpoint / "trainer_state.json"

    if state_file.exists():
        import json

        with open(state_file, "r") as f:
            state = json.load(f)

        print("Global step:", state.get("global_step"))
        print("Epoch:", state.get("epoch"))

print("\nAll checkpoint directories:")

for p in sorted(
    OUTPUT_DIR.glob("checkpoint-*"),
    key=lambda x: int(x.name.split("-")[-1])
):
    print(" ", p.name)


In [ ]:
from pathlib import Path

ckpt = Path(
    "/kaggle/working/MultiMend/"
    "models/multimend-codet5-small/checkpoint-581008"
)

print("Exists:", ckpt.exists())

if ckpt.exists():
    print("Files:")
    for p in ckpt.iterdir():
        print(" -", p.name)


In [ ]:
import shutil
from pathlib import Path

ckpt = Path(
    "/kaggle/working/MultiMend/"
    "models/multimend-codet5-small/checkpoint-581008"
)

zip_path = Path("/kaggle/working/checkpoint-581008")

shutil.make_archive(
    str(zip_path),
    "zip",
    root_dir=ckpt.parent,
    base_dir=ckpt.name
)

final_zip = Path("/kaggle/working/checkpoint-581008.zip")

print("Created:", final_zip)
print("Size:", round(final_zip.stat().st_size / 1024**3, 2), "GB")


In [ ]:
!kaggle config view


In [ ]:
from pathlib import Path

zip_path = Path("/kaggle/working/checkpoint-174300.zip")

print("Exists:", zip_path.exists())

if zip_path.exists():
    print("Path:", zip_path)
    print("Size:", round(zip_path.stat().st_size / 1024**2, 2), "MB")
